# Neuro-Symbolic Real Estate Advisory - Chuẩn hoá dữ liệu & xây tập luật Symbolic v2

Notebook này thực hiện quy trình:

```text
3 file Excel crawl BĐS
→ Gộp dữ liệu, bỏ nguồn/source khỏi output
→ Chuẩn hoá giá, diện tích, vị trí, pháp lý, tiện ích, cấu trúc nhà
→ Xây symbolic features chặt hơn
→ Xây rule base có trọng số, bằng chứng và cảnh báo rủi ro
→ Sinh triples cho Knowledge Graph
→ Test tư vấn và so sánh BĐS
```

Điểm khác bản trước:

- Không xuất các cột `source`, `source_name`, `source_file`, `source_sheet`.
- Rule symbolic chặt hơn: có `risk_flags`, `data_quality_score`, `rule_strength`, `evidence`.
- Có phân biệt `hard constraints` và `soft constraints` khi khuyến nghị.
- Giữ `URL` để đối chiếu tin đăng. Nếu muốn bỏ URL, xoá `url` khỏi `EXPORT_COLUMNS` ở cell lưu file.


In [ ]:
# Cell 1 - Cài thư viện và import
!pip -q install openpyxl

import os
import re
import json
import math
import unicodedata
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)


In [ ]:
# Cell 2 - Cấu hình đường dẫn dữ liệu
# Cách dùng trên Colab:
# 1) Upload 3 file Excel lên /content
# 2) Hoặc mount Google Drive và đặt file trong MyDrive

# from google.colab import drive
# drive.mount('/content/drive')

INPUT_FILES = [
    "Crawl_data_batdongsan.xlsx",
    "Crawl_data_nhatot.xlsx",
    "Crawl_data_refine.xlsx",
]

# Notebook sẽ tự tìm file trong các thư mục này
SEARCH_ROOTS = [
    Path("/content"),
    Path("/content/drive/MyDrive"),
]

# Các sheet hợp lệ trong file crawl
VALID_SHEETS = [
    "refined_apartments",
    "Batdongsan.com.vn",
    "nhatot.com",
]

OUTPUT_DIR = Path("/content/bds_symbolic_output_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# Cell 3 - Load dữ liệu Excel, nhưng KHÔNG đưa source vào output

def find_file(filename: str) -> Path | None:
    for root in SEARCH_ROOTS:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0]
    return None


def read_all_excel_files(input_files: List[str], valid_sheets: List[str]) -> pd.DataFrame:
    frames = []
    loaded_log = []

    for fname in input_files:
        path = find_file(fname)
        if path is None:
            print(f"Không tìm thấy file: {fname}")
            continue

        xls = pd.ExcelFile(path)
        for sheet in valid_sheets:
            if sheet not in xls.sheet_names:
                continue

            df = pd.read_excel(path, sheet_name=sheet)
            if len(df) == 0:
                continue

            # Không thêm source/source_name/source_file vào DataFrame output.
            frames.append(df)
            loaded_log.append({"file": fname, "sheet": sheet, "rows": len(df)})

    if len(frames) == 0:
        raise FileNotFoundError("Không load được dữ liệu. Hãy kiểm tra tên file hoặc đường dẫn upload.")

    raw = pd.concat(frames, ignore_index=True)
    print("Log load dữ liệu, chỉ để kiểm tra, không lưu vào output:")
    display(pd.DataFrame(loaded_log))
    print("Tổng số dòng raw trước chuẩn hóa:", len(raw))
    return raw

raw_df = read_all_excel_files(INPUT_FILES, VALID_SHEETS)
print("Các cột có trong dữ liệu:")
print(list(raw_df.columns))
display(raw_df.head(3))


In [ ]:
# Cell 4 - Hàm xử lý text, số, giá, diện tích

def clean_text(x: Any) -> str:
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()


def remove_accents(text: Any) -> str:
    text = str(text)
    text = text.replace("đ", "d").replace("Đ", "D")
    text = unicodedata.normalize("NFD", text)
    return "".join(ch for ch in text if unicodedata.category(ch) != "Mn")


def norm_text(text: Any) -> str:
    return remove_accents(clean_text(text)).lower()


def vn_float(x: Any) -> float:
    if x is None or pd.isna(x):
        return np.nan
    s = str(x).strip().replace(" ", "")
    # Nếu có cả dấu . và , thì giả định . là phân tách nghìn, , là thập phân
    if "." in s and "," in s:
        s = s.replace(".", "").replace(",", ".")
    else:
        s = s.replace(",", ".")
    s = re.sub(r"[^0-9.\-]", "", s)
    if s in ["", ".", "-"]:
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan


def first_number(text: Any) -> float:
    s = clean_text(text)
    m = re.search(r"(\d+(?:[\.,]\d+)?)", s)
    if not m:
        return np.nan
    return vn_float(m.group(1))


def all_numbers(text: Any) -> List[float]:
    s = clean_text(text)
    nums = []
    for m in re.finditer(r"(\d+(?:[\.,]\d+)?)", s):
        val = vn_float(m.group(1))
        if not pd.isna(val):
            nums.append(val)
    return nums


def parse_area_m2(raw_area: Any, text_all: str = "") -> float:
    """
    Chuẩn hóa diện tích về m2.
    Ưu tiên cột Dien_Tich, sau đó tìm trong mô tả.
    """
    if isinstance(raw_area, (int, float)) and not pd.isna(raw_area):
        val = float(raw_area)
        return val if 10 <= val <= 2000 else np.nan

    candidates = [raw_area, text_all]

    for text in candidates:
        s = norm_text(text).replace("㎡", "m2").replace("m²", "m2")

        # 76 m2, DT: 76m2, công nhận 95m2
        m = re.search(r"(?:dt|dien tich|cong nhan|cn)?\s*[:\-]?\s*(\d+(?:[\.,]\d+)?)\s*m2", s)
        if m:
            val = vn_float(m.group(1))
            if 10 <= val <= 2000:
                return val

        # 5x20 hoặc 5,7 x 13,3
        m = re.search(r"(\d+(?:[\.,]\d+)?)\s*(?:m)?\s*[x×]\s*(\d+(?:[\.,]\d+)?)\s*(?:m)?", s)
        if m:
            a, b = vn_float(m.group(1)), vn_float(m.group(2))
            if not pd.isna(a) and not pd.isna(b):
                area = a * b
                if 10 <= area <= 2000:
                    return area

        # ngang 3.5m dài 15m / rộng 4m dài 20m
        m = re.search(r"(?:ngang|rong)\s*(\d+(?:[\.,]\d+)?)\s*m?.{0,30}(?:dai|sau)\s*(\d+(?:[\.,]\d+)?)\s*m?", s)
        if m:
            a, b = vn_float(m.group(1)), vn_float(m.group(2))
            if not pd.isna(a) and not pd.isna(b):
                area = a * b
                if 10 <= area <= 2000:
                    return area

    return np.nan


def parse_price_billion(raw_price: Any, text_all: str = "", area_m2: float | None = None) -> float:
    """
    Chuẩn hóa giá về tỷ VNĐ.
    Hỗ trợ: 8,9 tỷ; 900 triệu; 120 triệu/m2; số dạng 5.95 trong refined data.
    """
    if isinstance(raw_price, (int, float)) and not pd.isna(raw_price):
        val = float(raw_price)
        return val if 0.05 <= val <= 500 else np.nan

    candidates = [raw_price, text_all]

    for text in candidates:
        s_raw = clean_text(text)
        s = norm_text(s_raw)
        if s == "" or any(k in s for k in ["thoa thuan", "lien he", "lien hệ"]):
            continue

        # 120 triệu/m2
        m = re.search(r"(\d+(?:[\.,]\d+)?)\s*trieu\s*/\s*m2", s)
        if m and area_m2 is not None and not pd.isna(area_m2):
            val = vn_float(m.group(1))
            price = val * area_m2 / 1000
            if 0.05 <= price <= 500:
                return price

        # 8,9 tỷ / 8.9 tỉ
        m = re.search(r"(\d+(?:[\.,]\d+)?)\s*(ty|tỷ|ti|tỉ)", s)
        if m:
            val = vn_float(m.group(1))
            if 0.05 <= val <= 500:
                return val

        # 900 triệu
        m = re.search(r"(\d+(?:[\.,]\d+)?)\s*trieu", s)
        if m:
            val = vn_float(m.group(1)) / 1000
            if 0.05 <= val <= 500:
                return val

    # Nếu raw_price chỉ là số dạng text, mặc định là tỷ
    val = first_number(raw_price)
    if not pd.isna(val) and 0.05 <= val <= 500:
        return val

    return np.nan


def parse_int_value(raw: Any, text_all: str = "", keywords: List[str] | None = None) -> float:
    """Parse số nguyên từ cột hoặc từ text theo keywords."""
    val = first_number(raw)
    if not pd.isna(val):
        return int(round(val))

    if keywords:
        s = norm_text(text_all)
        for kw in keywords:
            # Ví dụ: số phòng ngủ: 3 phòng, 3PN
            pattern = rf"(?:{kw})\s*[:\-]?\s*(\d+)"
            m = re.search(pattern, s)
            if m:
                return int(m.group(1))

        # Case 2PN, 3PN
        if any("phong ngu" in kw or "pn" in kw for kw in keywords):
            m = re.search(r"(\d+)\s*pn", s)
            if m:
                return int(m.group(1))

    return np.nan


def parse_meter(raw: Any, text_all: str = "", mode: str = "generic") -> float:
    val = first_number(raw)
    if not pd.isna(val) and 0 < val <= 100:
        return val

    s = norm_text(text_all)

    if mode == "frontage":
        patterns = [
            r"mat tien\s*[:\-]?\s*(\d+(?:[\.,]\d+)?)\s*m",
            r"ngang\s*(\d+(?:[\.,]\d+)?)\s*m",
            r"mt\s*(\d+(?:[\.,]\d+)?)\s*m",
        ]
    elif mode == "road":
        patterns = [
            r"duong\s*[:\-]?\s*(\d+(?:[\.,]\d+)?)\s*m",
            r"hem\s*(?:xe hoi|oto)?\s*(\d+(?:[\.,]\d+)?)\s*m",
            r"duong rong\s*(\d+(?:[\.,]\d+)?)\s*m",
        ]
    else:
        patterns = [r"(\d+(?:[\.,]\d+)?)\s*m"]

    for pat in patterns:
        m = re.search(pat, s)
        if m:
            val = vn_float(m.group(1))
            if not pd.isna(val) and 0 < val <= 100:
                return val

    return np.nan


In [ ]:
# Cell 5 - Chuẩn hóa vị trí, loại BĐS, pháp lý, tiện ích

DISTRICT_CANONICAL = {
    "quan 1": "Quận 1", "q1": "Quận 1", "q.1": "Quận 1",
    "quan 2": "Quận 2", "q2": "Quận 2", "q.2": "Quận 2",
    "quan 3": "Quận 3", "q3": "Quận 3", "q.3": "Quận 3",
    "quan 4": "Quận 4", "q4": "Quận 4", "q.4": "Quận 4",
    "quan 5": "Quận 5", "q5": "Quận 5", "q.5": "Quận 5",
    "quan 6": "Quận 6", "q6": "Quận 6", "q.6": "Quận 6",
    "quan 7": "Quận 7", "q7": "Quận 7", "q.7": "Quận 7",
    "quan 8": "Quận 8", "q8": "Quận 8", "q.8": "Quận 8",
    "quan 9": "Quận 9", "q9": "Quận 9", "q.9": "Quận 9",
    "quan 10": "Quận 10", "q10": "Quận 10", "q.10": "Quận 10",
    "quan 11": "Quận 11", "q11": "Quận 11", "q.11": "Quận 11",
    "quan 12": "Quận 12", "q12": "Quận 12", "q.12": "Quận 12",
    "binh thanh": "Bình Thạnh",
    "phu nhuan": "Phú Nhuận",
    "go vap": "Gò Vấp",
    "go vap ": "Gò Vấp",
    "tan binh": "Tân Bình",
    "tan phu": "Tân Phú",
    "binh tan": "Bình Tân",
    "thu duc": "Thủ Đức",
    "tp thu duc": "Thủ Đức",
    "thanh pho thu duc": "Thủ Đức",
    "nha be": "Nhà Bè",
    "binh chanh": "Bình Chánh",
    "hoc mon": "Hóc Môn",
    "cu chi": "Củ Chi",
    "can gio": "Cần Giờ",
}

CENTRAL_DISTRICTS = {"Quận 1", "Quận 3", "Quận 5", "Quận 10", "Bình Thạnh", "Phú Nhuận"}
DEVELOPING_DISTRICTS = {"Thủ Đức", "Quận 2", "Quận 7", "Quận 9", "Nhà Bè", "Bình Chánh"}
SUBURBAN_DISTRICTS = {"Hóc Môn", "Củ Chi", "Cần Giờ", "Bình Tân", "Quận 12"}


def extract_district(location_text: Any) -> str:
    s = norm_text(location_text)

    # Quận số
    m = re.search(r"\b(?:quan|q\.?|q)\s*(\d{1,2})\b", s)
    if m:
        return f"Quận {int(m.group(1))}"

    for key, val in sorted(DISTRICT_CANONICAL.items(), key=lambda x: -len(x[0])):
        if key in s:
            return val

    return "Không rõ"


def extract_ward(location_text: Any) -> str:
    s0 = clean_text(location_text)
    m = re.search(r"(Phường|P\.)\s*([^,\-]+)", s0, flags=re.IGNORECASE)
    if m:
        return clean_text(m.group(2))
    return "Không rõ"


def location_zone(district: str) -> str:
    if district in CENTRAL_DISTRICTS:
        return "central"
    if district in DEVELOPING_DISTRICTS:
        return "developing"
    if district in SUBURBAN_DISTRICTS:
        return "suburban"
    if district == "Không rõ":
        return "unknown"
    return "urban"


def infer_property_type(text_all: Any) -> str:
    s = norm_text(text_all)

    # Ưu tiên dấu hiệu mạnh trước để tránh nhầm "nhà mặt tiền gần chung cư" thành apartment.
    if any(k in s for k in ["dat nen", "lo dat", "dat "]):
        return "land"
    if any(k in s for k in ["biet thu", "villa"]):
        return "villa"
    if any(k in s for k in ["shophouse", "nha pho", "nha mat pho", "mat tien", "nha mt", "nha mat tien"]):
        return "townhouse"
    if any(k in s for k in ["nha rieng", "nha hem", "hem", "hxh", "nha "]):
        return "house"
    if any(k in s for k in ["can ho", "chung cu", "apartment", "vinhomes", "officetel"]):
        return "apartment"
    return "unknown"


def normalize_legal(raw_legal: Any, text_all: str = "") -> str:
    s = norm_text(str(raw_legal) + " " + str(text_all))
    if s.strip() == "":
        return "unknown"

    if any(k in s for k in ["so hong rieng", "shr", "so do", "so hong", "phap ly so", "so rieng"]):
        return "clear_ownership"
    if any(k in s for k in ["hop dong mua ban", "hdmb", "hop dong", "vi bang"]):
        return "contract_based"
    if any(k in s for k in ["dang cho so", "cho so", "chua so", "chua co so", "dang lam so"]):
        return "pending"
    if any(k in s for k in ["khong ro", "dang cap nhat", "nan"]):
        return "unknown"
    return "other"


def legal_score(legal_class: str) -> float:
    mapping = {
        "clear_ownership": 1.00,
        "contract_based": 0.60,
        "pending": 0.40,
        "other": 0.50,
        "unknown": 0.20,
    }
    return mapping.get(legal_class, 0.20)


AMENITY_PATTERNS = {
    "school": [r"truong hoc", r"truong quoc te", r"mam non", r"tieu hoc", r"thcs", r"thpt", r"dai hoc", r"cho con"],
    "hospital": [r"benh vien", r"phong kham", r"y te"],
    # Không dùng pattern "\bcho\b" vì dễ nhầm với cụm "cho con" trong truy vấn.
    "market": [r"gan cho", r"cho dan sinh", r"cho truyen thong", r"sieu thi", r"bach hoa", r"winmart", r"coopmart", r"bigc", r"aeon"],
    "park": [r"cong vien", r"cay xanh", r"noi bo xanh"],
    "mall": [r"trung tam thuong mai", r"vincom", r"lotte", r"mall", r"aeon"],
    "business": [r"kinh doanh", r"mat tien", r"shophouse", r"van phong", r"spa", r"cua hang", r"nha hang"],
    "rental": [r"cho thue", r"hd thue", r"hop dong thue", r"dong tien", r"khai thac thue"],
    "car_access": [r"xe hoi", r"oto", r"o to", r"hxh", r"hem xe hoi", r"duong rong"],
    "furnished": [r"noi that", r"full noi that", r"day du noi that", r"co ban"],
    "river_view": [r"view song", r"ven song", r"song sai gon", r"view kenh"],
    "urgent_sale": [r"ban gap", r"can tien", r"giam gia", r"cat lo", r"keo gia"],
    "planning_infra": [r"quy hoach", r"ha tang", r"metro", r"cao toc", r"san bay", r"thu thiem", r"vanh dai"],
    "security": [r"an ninh", r"bao ve", r"camera", r"khu dan tri", r"yên tĩnh", r"yen tinh"],
}


def extract_amenity_tags(text_all: Any) -> List[str]:
    s = norm_text(text_all)
    tags = []
    for tag, patterns in AMENITY_PATTERNS.items():
        if any(re.search(pat, s) for pat in patterns):
            tags.append(tag)
    return sorted(set(tags))


In [ ]:
# Cell 6 - Chuẩn hóa toàn bộ bảng dữ liệu

def get_col(df: pd.DataFrame, col: str, default: Any = "") -> pd.Series:
    if col in df.columns:
        return df[col]
    return pd.Series([default] * len(df), index=df.index)


def standardize_dataframe(raw: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()

    out["title"] = get_col(raw, "Tieu_De").apply(clean_text)
    out["description"] = get_col(raw, "Mo_Ta").apply(clean_text)
    out["url"] = get_col(raw, "URL").apply(clean_text)

    out["raw_location"] = get_col(raw, "Vi_Tri").apply(clean_text)
    out["raw_area"] = get_col(raw, "Dien_Tich").apply(clean_text)
    out["raw_price"] = get_col(raw, "Gia").apply(clean_text)
    out["raw_amenities"] = get_col(raw, "Tien_Ich").apply(clean_text)
    out["raw_legal"] = get_col(raw, "Phap_Ly").apply(clean_text)
    out["raw_furniture"] = get_col(raw, "Noi_That").apply(clean_text)

    out["text_all"] = (
        out["title"] + " " + out["description"] + " " + out["raw_amenities"] + " " +
        out["raw_location"] + " " + out["raw_legal"] + " " + out["raw_furniture"]
    ).apply(clean_text)

    out["area_m2"] = [parse_area_m2(a, t) for a, t in zip(out["raw_area"], out["text_all"])]
    out["price_billion"] = [parse_price_billion(p, t, a) for p, t, a in zip(out["raw_price"], out["text_all"], out["area_m2"])]

    out["price_per_m2_million"] = np.where(
        (out["area_m2"].notna()) & (out["area_m2"] > 0) & (out["price_billion"].notna()),
        out["price_billion"] * 1000 / out["area_m2"],
        np.nan,
    )

    out["bedrooms"] = [
        parse_int_value(v, t, keywords=["so phong ngu", "phong ngu", "pn"])
        for v, t in zip(get_col(raw, "So_PN"), out["text_all"])
    ]
    out["bathrooms"] = [
        parse_int_value(v, t, keywords=["so phong tam", "ve sinh", "wc", "phong tam"])
        for v, t in zip(get_col(raw, "So_PT"), out["text_all"])
    ]
    out["floors"] = [
        parse_int_value(v, t, keywords=["so tang", "tang"])
        for v, t in zip(get_col(raw, "So_Tang"), out["text_all"])
    ]

    out["frontage_m"] = [
        parse_meter(v, t, mode="frontage")
        for v, t in zip(get_col(raw, "Mat_Tien"), out["text_all"])
    ]
    out["road_width_m"] = [
        parse_meter(v, t, mode="road")
        for v, t in zip(get_col(raw, "Duong_Vao"), out["text_all"])
    ]

    out["district"] = out["raw_location"].apply(extract_district)
    out["ward"] = out["raw_location"].apply(extract_ward)
    out["location_zone"] = out["district"].apply(location_zone)
    out["property_type"] = out["text_all"].apply(infer_property_type)
    out["legal_class"] = [normalize_legal(l, t) for l, t in zip(out["raw_legal"], out["text_all"])]
    out["legal_score"] = out["legal_class"].apply(legal_score)
    out["amenity_tags"] = out["text_all"].apply(extract_amenity_tags)

    # ID ổn định, không dựa vào source
    out["property_id"] = [f"BDS_{i+1:05d}" for i in range(len(out))]

    return out

bds = standardize_dataframe(raw_df)
print("Sau chuẩn hóa ban đầu:", bds.shape)
display(bds.head(5))


In [ ]:
# Cell 7 - Khử trùng lặp và kiểm tra chất lượng dữ liệu

def make_fingerprint(row: pd.Series) -> str:
    title = norm_text(row.get("title", ""))[:80]
    loc = norm_text(row.get("raw_location", ""))[:80]
    price = row.get("price_billion", np.nan)
    area = row.get("area_m2", np.nan)
    price_key = "NA" if pd.isna(price) else str(round(float(price), 2))
    area_key = "NA" if pd.isna(area) else str(round(float(area), 1))
    return f"{title}|{loc}|{price_key}|{area_key}"


def clean_and_deduplicate(df: pd.DataFrame) -> pd.DataFrame:
    temp = df.copy()
    temp["url_norm"] = temp["url"].apply(lambda x: norm_text(x))
    temp["fingerprint"] = temp.apply(make_fingerprint, axis=1)

    # Ưu tiên dedup theo URL nếu có URL
    has_url = temp[temp["url_norm"] != ""].drop_duplicates("url_norm", keep="first")
    no_url = temp[temp["url_norm"] == ""]
    temp = pd.concat([has_url, no_url], ignore_index=True)

    # Dedup mềm theo fingerprint
    temp = temp.drop_duplicates("fingerprint", keep="first").reset_index(drop=True)

    # Reset ID sau dedup
    temp["property_id"] = [f"BDS_{i+1:05d}" for i in range(len(temp))]
    return temp

bds = clean_and_deduplicate(bds)
print("Sau khử trùng lặp:", bds.shape)


def validate_numeric_range(row: pd.Series) -> List[str]:
    flags = []
    price = row["price_billion"]
    area = row["area_m2"]
    ppm = row["price_per_m2_million"]

    if pd.isna(price):
        flags.append("missing_price")
    elif price < 0.2 or price > 300:
        flags.append("abnormal_price")

    if pd.isna(area):
        flags.append("missing_area")
    elif area < 10 or area > 1000:
        flags.append("abnormal_area")

    if pd.isna(ppm):
        flags.append("missing_price_per_m2")
    elif ppm < 5 or ppm > 1000:
        flags.append("abnormal_price_per_m2")

    if row["district"] == "Không rõ":
        flags.append("missing_district")

    if row["legal_class"] == "unknown":
        flags.append("missing_legal")

    if row["property_type"] == "unknown":
        flags.append("unknown_property_type")

    return flags


def data_quality_score(flags: List[str]) -> float:
    penalty_map = {
        "missing_price": 0.25,
        "abnormal_price": 0.30,
        "missing_area": 0.20,
        "abnormal_area": 0.25,
        "missing_price_per_m2": 0.10,
        "abnormal_price_per_m2": 0.20,
        "missing_district": 0.20,
        "missing_legal": 0.15,
        "unknown_property_type": 0.05,
    }
    score = 1.0
    for f in flags:
        score -= penalty_map.get(f, 0.05)
    return max(0.0, round(score, 3))

bds["data_flags"] = bds.apply(validate_numeric_range, axis=1)
bds["data_quality_score"] = bds["data_flags"].apply(data_quality_score)

display(bds[["property_id", "title", "district", "price_billion", "area_m2", "price_per_m2_million", "legal_class", "property_type", "data_quality_score", "data_flags"]].head(10))


In [ ]:
# Cell 8 - Symbolic bands và fuzzy membership

def price_band(price: float) -> str:
    if pd.isna(price): return "unknown"
    if price < 3: return "very_low"
    if price < 5: return "low"
    if price < 10: return "medium"
    if price < 20: return "medium_high"
    return "high"


def area_band(area: float) -> str:
    if pd.isna(area): return "unknown"
    if area < 50: return "small"
    if area < 90: return "medium"
    if area < 150: return "large"
    return "very_large"


def bedrooms_band(bedrooms: float) -> str:
    if pd.isna(bedrooms): return "unknown"
    if bedrooms <= 1: return "studio_or_1br"
    if bedrooms == 2: return "2br"
    if bedrooms == 3: return "3br"
    return "4br_plus"


def road_band(road_width: float) -> str:
    if pd.isna(road_width): return "unknown"
    if road_width < 3: return "small_alley"
    if road_width < 5: return "motorbike_alley"
    if road_width < 8: return "car_access"
    return "wide_road"


def frontage_band(frontage: float) -> str:
    if pd.isna(frontage): return "unknown"
    if frontage < 3.5: return "narrow"
    if frontage < 5: return "standard"
    if frontage < 8: return "wide"
    return "very_wide"


def trapezoid(x: float, a: float, b: float, c: float, d: float) -> float:
    """Membership function hình thang."""
    if pd.isna(x): return 0.0
    if x <= a or x >= d: return 0.0
    if b <= x <= c: return 1.0
    if a < x < b: return (x - a) / (b - a)
    if c < x < d: return (d - x) / (d - c)
    return 0.0


def has_tag(tags: List[str], tag: str) -> bool:
    return isinstance(tags, list) and tag in tags

bds["price_band"] = bds["price_billion"].apply(price_band)
bds["area_band"] = bds["area_m2"].apply(area_band)
bds["bedrooms_band"] = bds["bedrooms"].apply(bedrooms_band)
bds["road_band"] = bds["road_width_m"].apply(road_band)
bds["frontage_band"] = bds["frontage_m"].apply(frontage_band)

# Median giá/m2 theo quận để đánh giá giá hợp lý.
global_median_ppm = bds["price_per_m2_million"].median()
bds["district_median_price_per_m2"] = bds.groupby("district")["price_per_m2_million"].transform("median").fillna(global_median_ppm)


def price_reasonable_score(row: pd.Series) -> float:
    ppm = row["price_per_m2_million"]
    med = row["district_median_price_per_m2"]
    if pd.isna(ppm) or pd.isna(med) or med <= 0:
        return 0.50
    ratio = ppm / med
    if ratio <= 0.85:
        return 1.00
    if ratio <= 1.00:
        return 0.90
    if ratio <= 1.20:
        return 0.70
    if ratio <= 1.50:
        return 0.40
    return 0.15

bds["price_reasonable_score"] = bds.apply(price_reasonable_score, axis=1)

display(bds[["property_id", "district", "price_billion", "price_band", "area_m2", "area_band", "bedrooms_band", "road_band", "frontage_band", "price_reasonable_score"]].head(10))


In [ ]:
# Cell 9 - Điểm symbolic nâng cao: gia đình, kinh doanh, cho thuê, đầu tư, rủi ro

def family_suitability(row: pd.Series) -> float:
    score = 0.0
    if not pd.isna(row["bedrooms"]):
        if row["bedrooms"] >= 3: score += 0.25
        elif row["bedrooms"] == 2: score += 0.20
        elif row["bedrooms"] == 1: score += 0.08
    if not pd.isna(row["bathrooms"]):
        if row["bathrooms"] >= 2: score += 0.15
        else: score += 0.06
    if not pd.isna(row["area_m2"]):
        if row["area_m2"] >= 70: score += 0.20
        elif row["area_m2"] >= 50: score += 0.15
        else: score += 0.05
    if row["legal_score"] >= 0.8: score += 0.18
    if row["location_zone"] in ["central", "urban", "developing"]: score += 0.07
    if has_tag(row["amenity_tags"], "school"): score += 0.08
    if has_tag(row["amenity_tags"], "market"): score += 0.04
    if has_tag(row["amenity_tags"], "hospital"): score += 0.03
    if has_tag(row["amenity_tags"], "security"): score += 0.05
    return round(min(score, 1.0), 3)


def business_potential(row: pd.Series) -> float:
    score = 0.0
    if row["property_type"] in ["townhouse", "villa"]: score += 0.18
    if has_tag(row["amenity_tags"], "business"): score += 0.22
    if not pd.isna(row["frontage_m"]):
        if row["frontage_m"] >= 6: score += 0.22
        elif row["frontage_m"] >= 4: score += 0.16
        elif row["frontage_m"] >= 3.5: score += 0.08
    if not pd.isna(row["road_width_m"]):
        if row["road_width_m"] >= 8: score += 0.20
        elif row["road_width_m"] >= 5: score += 0.14
        elif row["road_width_m"] >= 3: score += 0.05
    if has_tag(row["amenity_tags"], "car_access"): score += 0.10
    if row["location_zone"] in ["central", "urban"]: score += 0.08
    return round(min(score, 1.0), 3)


def rental_potential(row: pd.Series) -> float:
    score = 0.0
    if row["property_type"] in ["apartment", "townhouse", "house"]: score += 0.15
    if has_tag(row["amenity_tags"], "rental"): score += 0.25
    if row["location_zone"] == "central": score += 0.22
    elif row["location_zone"] in ["urban", "developing"]: score += 0.16
    if not pd.isna(row["bedrooms"]):
        if 1 <= row["bedrooms"] <= 3: score += 0.12
        elif row["bedrooms"] >= 4: score += 0.08
    if row["legal_score"] >= 0.8: score += 0.12
    if has_tag(row["amenity_tags"], "furnished"): score += 0.06
    if has_tag(row["amenity_tags"], "mall") or has_tag(row["amenity_tags"], "market"): score += 0.05
    if row["price_reasonable_score"] >= 0.7: score += 0.05
    return round(min(score, 1.0), 3)


def investment_potential(row: pd.Series) -> float:
    score = 0.0
    if row["location_zone"] == "central": score += 0.25
    elif row["location_zone"] == "developing": score += 0.22
    elif row["location_zone"] == "urban": score += 0.16
    if row["legal_score"] >= 0.8: score += 0.18
    elif row["legal_score"] >= 0.6: score += 0.10
    score += 0.22 * row["price_reasonable_score"]
    if has_tag(row["amenity_tags"], "planning_infra"): score += 0.14
    if has_tag(row["amenity_tags"], "urgent_sale"): score += 0.06
    if row["property_type"] in ["townhouse", "apartment", "land"]: score += 0.08
    if row["data_quality_score"] >= 0.8: score += 0.07
    return round(min(score, 1.0), 3)


def risk_score(row: pd.Series) -> float:
    risk = 0.0
    if row["legal_class"] == "unknown": risk += 0.25
    elif row["legal_class"] in ["pending", "contract_based"]: risk += 0.15
    if row["data_quality_score"] < 0.6: risk += 0.25
    elif row["data_quality_score"] < 0.8: risk += 0.10
    if row["price_reasonable_score"] < 0.4: risk += 0.15
    if "abnormal_price_per_m2" in row["data_flags"]: risk += 0.20
    if "missing_district" in row["data_flags"]: risk += 0.10
    return round(min(risk, 1.0), 3)

bds["family_suitability_score"] = bds.apply(family_suitability, axis=1)
bds["business_potential_score"] = bds.apply(business_potential, axis=1)
bds["rental_potential_score"] = bds.apply(rental_potential, axis=1)
bds["investment_potential_score"] = bds.apply(investment_potential, axis=1)
bds["risk_score"] = bds.apply(risk_score, axis=1)


def symbolic_level(score: float) -> str:
    if pd.isna(score): return "unknown"
    if score >= 0.75: return "high"
    if score >= 0.50: return "medium"
    if score >= 0.25: return "low"
    return "very_low"

for col in [
    "family_suitability_score", "business_potential_score", "rental_potential_score",
    "investment_potential_score", "risk_score", "data_quality_score"
]:
    bds[col.replace("_score", "_level")] = bds[col].apply(symbolic_level)

display(bds[["property_id", "property_type", "district", "legal_class", "family_suitability_score", "business_potential_score", "rental_potential_score", "investment_potential_score", "risk_score", "data_quality_score"]].head(10))


In [ ]:
# Cell 10 - Xây tập luật Symbolic chặt hơn: có điều kiện, kết luận, strength, evidence

RULE_BASE = [
    {
        "rule_id": "R01_LEGAL_CLEAR",
        "target": "legal_risk",
        "conclusion": "low_legal_risk",
        "description": "Pháp lý rõ khi có sổ đỏ/sổ hồng/sổ hồng riêng.",
    },
    {
        "rule_id": "R02_LEGAL_WARNING",
        "target": "risk_warning",
        "conclusion": "need_legal_verification",
        "description": "Pháp lý không rõ, đang chờ sổ hoặc hợp đồng mua bán cần kiểm tra thêm.",
    },
    {
        "rule_id": "R03_FAMILY_STRONG",
        "target": "suitable_for",
        "conclusion": "family",
        "description": "Phù hợp gia đình nếu đủ phòng, diện tích hợp lý, pháp lý tốt và tiện ích sống tốt.",
    },
    {
        "rule_id": "R04_BUSINESS_STRONG",
        "target": "suitable_for",
        "conclusion": "business",
        "description": "Phù hợp kinh doanh nếu có mặt tiền/đường lớn/từ khóa kinh doanh.",
    },
    {
        "rule_id": "R05_RENTAL_STRONG",
        "target": "suitable_for",
        "conclusion": "rental",
        "description": "Phù hợp cho thuê nếu vị trí tốt, loại hình dễ cho thuê và có tín hiệu khai thác thuê.",
    },
    {
        "rule_id": "R06_INVESTMENT_STRONG",
        "target": "suitable_for",
        "conclusion": "investment",
        "description": "Phù hợp đầu tư nếu vị trí tốt, pháp lý rõ và giá/m2 hợp lý so với khu vực.",
    },
    {
        "rule_id": "R07_DATA_LOW_CONFIDENCE",
        "target": "risk_warning",
        "conclusion": "low_data_confidence",
        "description": "Dữ liệu thiếu nhiều trường quan trọng làm giảm độ tin cậy khuyến nghị.",
    },
    {
        "rule_id": "R08_OVERPRICED_WARNING",
        "target": "risk_warning",
        "conclusion": "possibly_overpriced",
        "description": "Giá/m2 cao hơn đáng kể so với trung vị khu vực.",
    },
]


def apply_rule_engine(row: pd.Series) -> Tuple[List[str], List[Dict[str, Any]], List[str]]:
    facts = []
    evidence = []
    risk_flags = []

    # R01
    if row["legal_class"] == "clear_ownership":
        facts.append("low_legal_risk")
        evidence.append({
            "rule_id": "R01_LEGAL_CLEAR",
            "strength": 1.0,
            "evidence": f"legal_class={row['legal_class']}",
        })

    # R02
    if row["legal_class"] in ["unknown", "pending", "contract_based", "other"]:
        strength = 0.9 if row["legal_class"] == "unknown" else 0.7
        risk_flags.append("need_legal_verification")
        evidence.append({
            "rule_id": "R02_LEGAL_WARNING",
            "strength": strength,
            "evidence": f"legal_class={row['legal_class']}",
        })

    # R03
    if row["family_suitability_score"] >= 0.70:
        facts.append("suitable_for_family")
        evidence.append({
            "rule_id": "R03_FAMILY_STRONG",
            "strength": float(row["family_suitability_score"]),
            "evidence": f"bedrooms={row['bedrooms']}, area={row['area_m2']}, legal_score={row['legal_score']}",
        })

    # R04
    if row["business_potential_score"] >= 0.65:
        facts.append("suitable_for_business")
        evidence.append({
            "rule_id": "R04_BUSINESS_STRONG",
            "strength": float(row["business_potential_score"]),
            "evidence": f"frontage={row['frontage_m']}, road={row['road_width_m']}, tags={row['amenity_tags']}",
        })

    # R05
    if row["rental_potential_score"] >= 0.65:
        facts.append("suitable_for_rental")
        evidence.append({
            "rule_id": "R05_RENTAL_STRONG",
            "strength": float(row["rental_potential_score"]),
            "evidence": f"zone={row['location_zone']}, type={row['property_type']}, tags={row['amenity_tags']}",
        })

    # R06
    if row["investment_potential_score"] >= 0.70:
        facts.append("suitable_for_investment")
        evidence.append({
            "rule_id": "R06_INVESTMENT_STRONG",
            "strength": float(row["investment_potential_score"]),
            "evidence": f"zone={row['location_zone']}, legal_score={row['legal_score']}, price_reasonable={row['price_reasonable_score']}",
        })

    # R07
    if row["data_quality_score"] < 0.70:
        risk_flags.append("low_data_confidence")
        evidence.append({
            "rule_id": "R07_DATA_LOW_CONFIDENCE",
            "strength": round(1 - row["data_quality_score"], 3),
            "evidence": f"data_flags={row['data_flags']}",
        })

    # R08
    if row["price_reasonable_score"] < 0.40:
        risk_flags.append("possibly_overpriced")
        evidence.append({
            "rule_id": "R08_OVERPRICED_WARNING",
            "strength": round(1 - row["price_reasonable_score"], 3),
            "evidence": f"price_per_m2={row['price_per_m2_million']}, district_median={row['district_median_price_per_m2']}",
        })

    return sorted(set(facts)), evidence, sorted(set(risk_flags))

rule_outputs = bds.apply(apply_rule_engine, axis=1)
bds["symbolic_facts"] = [x[0] for x in rule_outputs]
bds["rule_evidence"] = [x[1] for x in rule_outputs]
bds["risk_flags"] = [x[2] for x in rule_outputs]
bds["triggered_rules"] = [[ev["rule_id"] for ev in evs] for evs in bds["rule_evidence"]]

display(bds[["property_id", "symbolic_facts", "risk_flags", "triggered_rules", "rule_evidence"]].head(10))


In [ ]:
# Cell 11 - Ma trận tương đồng vị trí để hỗ trợ Spreading Activation

LOCATION_SIMILARITY_MANUAL = {
    ("Quận 3", "Quận 1"): 0.90,
    ("Quận 3", "Quận 10"): 0.85,
    ("Quận 3", "Phú Nhuận"): 0.75,
    ("Quận 1", "Bình Thạnh"): 0.75,
    ("Quận 1", "Quận 4"): 0.75,
    ("Quận 10", "Quận 5"): 0.80,
    ("Quận 10", "Quận 11"): 0.75,
    ("Bình Thạnh", "Phú Nhuận"): 0.75,
    ("Bình Thạnh", "Thủ Đức"): 0.60,
    ("Quận 2", "Thủ Đức"): 0.75,
    ("Quận 7", "Nhà Bè"): 0.75,
    ("Bình Tân", "Tân Phú"): 0.70,
    ("Tân Bình", "Phú Nhuận"): 0.75,
    ("Gò Vấp", "Phú Nhuận"): 0.65,
    ("Gò Vấp", "Tân Bình"): 0.65,
}


def district_similarity(a: str, b: str) -> float:
    if a == b:
        return 1.0
    if a == "Không rõ" or b == "Không rõ":
        return 0.15
    if (a, b) in LOCATION_SIMILARITY_MANUAL:
        return LOCATION_SIMILARITY_MANUAL[(a, b)]
    if (b, a) in LOCATION_SIMILARITY_MANUAL:
        return LOCATION_SIMILARITY_MANUAL[(b, a)]

    za, zb = location_zone(a), location_zone(b)
    if za == zb and za != "unknown":
        return 0.60
    if {za, zb} == {"central", "urban"}:
        return 0.50
    if {za, zb} == {"central", "developing"}:
        return 0.42
    if {za, zb} == {"developing", "urban"}:
        return 0.45
    if {za, zb} == {"developing", "suburban"}:
        return 0.35
    return 0.25


districts = sorted(bds["district"].dropna().unique().tolist())
location_similarity_matrix = pd.DataFrame(index=districts, columns=districts, dtype=float)
for a in districts:
    for b in districts:
        location_similarity_matrix.loc[a, b] = district_similarity(a, b)

display(location_similarity_matrix.head())


In [ ]:
# Cell 12 - Sinh triples cho Knowledge Graph, không có source

def create_triples(df: pd.DataFrame) -> pd.DataFrame:
    triples = []
    numeric_predicates = {
        "price_billion": "hasPriceBillion",
        "area_m2": "hasAreaM2",
        "price_per_m2_million": "hasPricePerM2Million",
        "bedrooms": "hasBedrooms",
        "bathrooms": "hasBathrooms",
        "floors": "hasFloors",
        "frontage_m": "hasFrontageM",
        "road_width_m": "hasRoadWidthM",
        "legal_score": "hasLegalScore",
        "data_quality_score": "hasDataQualityScore",
        "risk_score": "hasRiskScore",
        "family_suitability_score": "hasFamilySuitabilityScore",
        "business_potential_score": "hasBusinessPotentialScore",
        "rental_potential_score": "hasRentalPotentialScore",
        "investment_potential_score": "hasInvestmentPotentialScore",
    }

    for _, row in df.iterrows():
        s = row["property_id"]
        triples.append([s, "type", "Property"])
        triples.append([s, "hasPropertyType", row["property_type"]])
        triples.append([s, "locatedInDistrict", row["district"]])
        triples.append([s, "locatedInWard", row["ward"]])
        triples.append([s, "locatedInZone", row["location_zone"]])
        triples.append([s, "hasPriceBand", row["price_band"]])
        triples.append([s, "hasAreaBand", row["area_band"]])
        triples.append([s, "hasBedroomBand", row["bedrooms_band"]])
        triples.append([s, "hasRoadBand", row["road_band"]])
        triples.append([s, "hasFrontageBand", row["frontage_band"]])
        triples.append([s, "hasLegalClass", row["legal_class"]])

        for col, pred in numeric_predicates.items():
            val = row.get(col, np.nan)
            if not pd.isna(val):
                triples.append([s, pred, float(val)])

        for tag in row["amenity_tags"]:
            triples.append([s, "hasAmenityTag", tag])

        for fact in row["symbolic_facts"]:
            triples.append([s, "inferredFact", fact])

        for flag in row["risk_flags"]:
            triples.append([s, "hasRiskFlag", flag])

        for rule in row["triggered_rules"]:
            triples.append([s, "triggeredRule", rule])

    return pd.DataFrame(triples, columns=["subject", "predicate", "object"])

kg_triples = create_triples(bds)
print("Số triples:", len(kg_triples))
display(kg_triples.head(25))


In [ ]:
# Cell 13 - Parse truy vấn người dùng thành symbolic profile

def parse_user_query(query: str) -> Dict[str, Any]:
    q_raw = clean_text(query)
    q = norm_text(query)

    profile = {
        "raw_query": q_raw,
        "budget_billion": None,
        "target_area_m2": None,
        "district": None,
        "bedrooms_min": None,
        "bathrooms_min": None,
        "legal_required": False,
        "desired_amenities": [],
        "preferred_property_type": None,
        "purpose": "general",
        "hard_constraints": [],
        "soft_constraints": [],
    }

    # Ngân sách
    m = re.search(r"(\d+(?:[\.,]\d+)?)\s*(ty|tỷ|ti|tỉ)", q_raw.lower())
    if m:
        profile["budget_billion"] = vn_float(m.group(1))
        profile["soft_constraints"].append("budget")

    # Diện tích
    m = re.search(r"(\d+(?:[\.,]\d+)?)\s*(m2|m²)", q_raw.lower())
    if m:
        profile["target_area_m2"] = vn_float(m.group(1))
        profile["soft_constraints"].append("area")

    # Số phòng ngủ / phòng tắm
    m = re.search(r"(\d+)\s*(pn|phong ngu|phòng ngủ)", q)
    if m:
        profile["bedrooms_min"] = int(m.group(1))
        profile["hard_constraints"].append("bedrooms_min")

    m = re.search(r"(\d+)\s*(wc|phong tam|phòng tắm|ve sinh)", q)
    if m:
        profile["bathrooms_min"] = int(m.group(1))
        profile["soft_constraints"].append("bathrooms_min")

    # Vị trí
    district = extract_district(q_raw)
    if district != "Không rõ":
        profile["district"] = district
        profile["soft_constraints"].append("location")

    # Pháp lý
    if any(k in q for k in ["phap ly ro", "so hong", "so do", "so rieng", "an toan phap ly", "pháp lý rõ"]):
        profile["legal_required"] = True
        profile["hard_constraints"].append("legal_required")

    # Loại BĐS
    if any(k in q for k in ["can ho", "chung cu", "apartment"]):
        profile["preferred_property_type"] = "apartment"
    elif any(k in q for k in ["nha pho", "mat tien", "mat pho", "shophouse"]):
        profile["preferred_property_type"] = "townhouse"
    elif any(k in q for k in ["biet thu", "villa"]):
        profile["preferred_property_type"] = "villa"
    elif any(k in q for k in ["nha rieng", "nha hem", "hxh"]):
        profile["preferred_property_type"] = "house"

    if profile["preferred_property_type"]:
        profile["soft_constraints"].append("property_type")

    # Mục đích
    if any(k in q for k in ["kinh doanh", "mat tien", "mo cua hang", "van phong", "shop"]):
        profile["purpose"] = "business"
    elif any(k in q for k in ["dau tu", "tang gia", "sinh loi", "tiem nang", "lướt sóng"]):
        profile["purpose"] = "investment"
    elif any(k in q for k in ["cho thue", "dong tien", "khai thac thue"]):
        profile["purpose"] = "rental"
    elif any(k in q for k in ["o", "gia dinh", "cho con", "an cu"]):
        profile["purpose"] = "family"

    # Tags tiện ích
    desired = []
    for tag, patterns in AMENITY_PATTERNS.items():
        if any(re.search(pat, q) for pat in patterns):
            desired.append(tag)
    profile["desired_amenities"] = sorted(set(desired))
    if desired:
        profile["soft_constraints"].append("amenity")

    return profile

example_query = "Tìm nhà quanh quận 3, khoảng 8 tỷ, 2PN trở lên, gần trường học cho con, pháp lý rõ ràng"
profile = parse_user_query(example_query)
print(json.dumps(profile, ensure_ascii=False, indent=2))


In [ ]:
# Cell 14 - Khuyến nghị bằng hard filter + fuzzy symbolic scoring

PURPOSE_SCORE_COL = {
    "family": "family_suitability_score",
    "business": "business_potential_score",
    "rental": "rental_potential_score",
    "investment": "investment_potential_score",
    "general": None,
}

WEIGHTS = {
    "family":      {"price": 0.24, "location": 0.22, "legal": 0.18, "area": 0.10, "amenity": 0.10, "type": 0.04, "purpose": 0.12},
    "business":    {"price": 0.18, "location": 0.22, "legal": 0.14, "area": 0.06, "amenity": 0.08, "type": 0.07, "purpose": 0.25},
    "rental":      {"price": 0.18, "location": 0.24, "legal": 0.14, "area": 0.07, "amenity": 0.08, "type": 0.07, "purpose": 0.22},
    "investment":  {"price": 0.24, "location": 0.20, "legal": 0.17, "area": 0.05, "amenity": 0.05, "type": 0.04, "purpose": 0.25},
    "general":     {"price": 0.23, "location": 0.22, "legal": 0.16, "area": 0.08, "amenity": 0.08, "type": 0.05, "purpose": 0.18},
}


def fuzzy_budget_score(price: float, budget: float | None) -> float:
    if budget is None or pd.isna(price):
        return 0.60
    if price <= budget:
        # Rẻ hơn ngân sách vẫn tốt, nhưng quá rẻ thì giảm nhẹ vì có thể khác phân khúc.
        ratio = price / budget
        if ratio >= 0.75: return 1.00
        if ratio >= 0.50: return 0.85
        return 0.70
    over = (price - budget) / budget
    if over <= 0.05: return 0.90
    if over <= 0.10: return 0.80
    if over <= 0.20: return 0.55
    if over <= 0.35: return 0.30
    return 0.05


def fuzzy_area_score(area: float, target_area: float | None) -> float:
    if target_area is None or pd.isna(area):
        return 0.60
    diff = abs(area - target_area) / max(target_area, 1)
    if diff <= 0.10: return 1.00
    if diff <= 0.25: return 0.80
    if diff <= 0.50: return 0.50
    return 0.20


def location_match_score(row_district: str, desired_district: str | None) -> float:
    if desired_district is None:
        return 0.60
    return district_similarity(row_district, desired_district)


def amenity_match_score(row_tags: List[str], desired_tags: List[str]) -> float:
    if not desired_tags:
        return 0.60
    if not isinstance(row_tags, list):
        return 0.0
    matched = len(set(row_tags) & set(desired_tags))
    return matched / len(set(desired_tags))


def property_type_score(row_type: str, preferred_type: str | None) -> float:
    if preferred_type is None:
        return 0.60
    if row_type == preferred_type:
        return 1.00
    # Gần nhau tương đối
    similar = {
        ("house", "townhouse"): 0.70,
        ("townhouse", "house"): 0.70,
        ("villa", "house"): 0.60,
        ("house", "villa"): 0.60,
    }
    return similar.get((row_type, preferred_type), 0.25)


def purpose_match_score(row: pd.Series, purpose: str) -> float:
    col = PURPOSE_SCORE_COL.get(purpose)
    if col is None:
        return max(row["family_suitability_score"], row["business_potential_score"], row["rental_potential_score"], row["investment_potential_score"])
    return row[col]


def hard_constraint_pass(row: pd.Series, profile: Dict[str, Any], relax_level: int = 0) -> bool:
    """
    relax_level = 0: chặt nhất
    relax_level = 1: nới ngân sách và phòng ngủ nhẹ
    relax_level = 2: nới pháp lý từ clear sang contract/pending nhưng sẽ bị phạt điểm
    """
    if profile.get("legal_required"):
        if relax_level < 2 and row["legal_class"] != "clear_ownership":
            return False
        if relax_level >= 2 and row["legal_score"] < 0.5:
            return False

    bedrooms_min = profile.get("bedrooms_min")
    if bedrooms_min is not None:
        # Chặt hơn bản cũ: nếu user yêu cầu số phòng ngủ mà tin đăng thiếu phòng ngủ,
        # relax_level 0 sẽ loại; relax_level >= 1 mới giữ lại nhưng điểm mục đích vẫn bị ảnh hưởng bởi thiếu dữ liệu.
        if pd.isna(row["bedrooms"]):
            if relax_level == 0:
                return False
        else:
            required = bedrooms_min if relax_level == 0 else max(1, bedrooms_min - 1)
            if row["bedrooms"] < required:
                return False

    return True


def score_property(row: pd.Series, profile: Dict[str, Any]) -> Tuple[float, Dict[str, float]]:
    detail = {}
    detail["price"] = fuzzy_budget_score(row["price_billion"], profile.get("budget_billion"))
    detail["location"] = location_match_score(row["district"], profile.get("district"))
    detail["legal"] = row["legal_score"] if profile.get("legal_required") else (0.65 + 0.35 * row["legal_score"])
    detail["area"] = fuzzy_area_score(row["area_m2"], profile.get("target_area_m2"))
    detail["amenity"] = amenity_match_score(row["amenity_tags"], profile.get("desired_amenities", []))
    detail["type"] = property_type_score(row["property_type"], profile.get("preferred_property_type"))
    detail["purpose"] = purpose_match_score(row, profile.get("purpose", "general"))

    weights = WEIGHTS.get(profile.get("purpose", "general"), WEIGHTS["general"])
    base = sum(detail[k] * weights[k] for k in weights)

    # Phạt rủi ro và dữ liệu thiếu.
    penalty = 1.0 - 0.22 * row["risk_score"]
    quality_boost = 0.85 + 0.15 * row["data_quality_score"]
    final = base * penalty * quality_boost

    return round(float(final), 4), {k: round(float(v), 4) for k, v in detail.items()}


def explain_row(row: pd.Series, profile: Dict[str, Any], detail: Dict[str, float]) -> str:
    parts = []
    if profile.get("district"):
        parts.append(f"Vị trí {row['district']} đạt điểm tương đồng {detail['location']:.2f} với {profile['district']}.")
    if profile.get("budget_billion") and not pd.isna(row["price_billion"]):
        parts.append(f"Giá {row['price_billion']:.2f} tỷ so với ngân sách {profile['budget_billion']:.2f} tỷ, điểm giá {detail['price']:.2f}.")
    if row["legal_class"] == "clear_ownership":
        parts.append("Pháp lý rõ, thuộc nhóm sổ đỏ/sổ hồng.")
    else:
        parts.append(f"Pháp lý thuộc nhóm {row['legal_class']}, cần kiểm tra thêm.")
    if detail["purpose"] >= 0.70:
        parts.append(f"Mục đích {profile.get('purpose')} phù hợp mạnh, điểm {detail['purpose']:.2f}.")
    elif detail["purpose"] >= 0.50:
        parts.append(f"Mục đích {profile.get('purpose')} phù hợp ở mức trung bình, điểm {detail['purpose']:.2f}.")
    if row["risk_flags"]:
        parts.append("Cảnh báo: " + ", ".join(row["risk_flags"]) + ".")
    if row["amenity_tags"]:
        parts.append("Tag nổi bật: " + ", ".join(row["amenity_tags"]) + ".")
    return " ".join(parts)


def recommend(query: str, df: pd.DataFrame, top_k: int = 5) -> Tuple[Dict[str, Any], pd.DataFrame]:
    profile = parse_user_query(query)

    # Thử từ chặt đến nới lỏng để tránh không có kết quả.
    candidate_df = None
    used_relax_level = 0
    for relax in [0, 1, 2]:
        mask = df.apply(lambda r: hard_constraint_pass(r, profile, relax_level=relax), axis=1)
        candidate_df = df[mask].copy()
        if len(candidate_df) > 0:
            used_relax_level = relax
            break

    results = []
    for _, row in candidate_df.iterrows():
        final, detail = score_property(row, profile)
        results.append({
            "property_id": row["property_id"],
            "final_score": final,
            "price_score": detail["price"],
            "location_score": detail["location"],
            "legal_score_used": detail["legal"],
            "area_score": detail["area"],
            "amenity_score": detail["amenity"],
            "type_score": detail["type"],
            "purpose_score": detail["purpose"],
            "property_type": row["property_type"],
            "title": row["title"],
            "district": row["district"],
            "ward": row["ward"],
            "price_billion": row["price_billion"],
            "area_m2": row["area_m2"],
            "price_per_m2_million": row["price_per_m2_million"],
            "bedrooms": row["bedrooms"],
            "bathrooms": row["bathrooms"],
            "legal_class": row["legal_class"],
            "amenity_tags": row["amenity_tags"],
            "symbolic_facts": row["symbolic_facts"],
            "risk_flags": row["risk_flags"],
            "triggered_rules": row["triggered_rules"],
            "data_quality_score": row["data_quality_score"],
            "explanation": explain_row(row, profile, detail),
            "url": row["url"],
        })

    out = pd.DataFrame(results)
    if len(out) > 0:
        out = out.sort_values("final_score", ascending=False).head(top_k).reset_index(drop=True)
    profile["used_relax_level"] = used_relax_level
    profile["num_candidates"] = len(candidate_df)
    return profile, out


In [ ]:
# Cell 15 - Test truy vấn tư vấn BĐS

query = "Tìm nhà quanh quận 3, khoảng 8 tỷ, 2PN trở lên, gần trường học cho con, pháp lý rõ ràng"
profile, recs = recommend(query, bds, top_k=5)

print("QUERY:", query)
print("\nSYMBOLIC PROFILE:")
print(json.dumps(profile, ensure_ascii=False, indent=2))

print("\nTOP RESULTS:")
display(recs[[
    "property_id", "final_score", "property_type", "district", "price_billion", "area_m2",
    "bedrooms", "legal_class", "purpose_score", "risk_flags", "triggered_rules"
]])

for i, row in recs.iterrows():
    print("=" * 120)
    print(f"TOP {i+1} | {row['property_id']} | Score={row['final_score']}")
    print("Title:", row["title"])
    print("Explain:", row["explanation"])
    print("URL:", row["url"])


In [ ]:
# Cell 16 - Module so sánh 2 BĐS dựa trên Symbolic factors

def compare_properties(property_id_1: str, property_id_2: str, df: pd.DataFrame = bds) -> pd.DataFrame:
    a = df[df["property_id"] == property_id_1]
    b = df[df["property_id"] == property_id_2]
    if len(a) == 0:
        raise ValueError(f"Không tìm thấy {property_id_1}")
    if len(b) == 0:
        raise ValueError(f"Không tìm thấy {property_id_2}")
    a = a.iloc[0]
    b = b.iloc[0]

    fields = [
        ("Loại BĐS", "property_type"),
        ("Quận", "district"),
        ("Phường", "ward"),
        ("Nhóm vị trí", "location_zone"),
        ("Giá tỷ", "price_billion"),
        ("Diện tích m2", "area_m2"),
        ("Giá triệu/m2", "price_per_m2_million"),
        ("Số phòng ngủ", "bedrooms"),
        ("Số phòng tắm", "bathrooms"),
        ("Số tầng", "floors"),
        ("Mặt tiền m", "frontage_m"),
        ("Đường vào m", "road_width_m"),
        ("Pháp lý", "legal_class"),
        ("Điểm gia đình", "family_suitability_score"),
        ("Điểm kinh doanh", "business_potential_score"),
        ("Điểm cho thuê", "rental_potential_score"),
        ("Điểm đầu tư", "investment_potential_score"),
        ("Điểm rủi ro", "risk_score"),
        ("Độ tin cậy dữ liệu", "data_quality_score"),
        ("Symbolic facts", "symbolic_facts"),
        ("Risk flags", "risk_flags"),
    ]

    rows = []
    for label, col in fields:
        rows.append({
            "Tiêu chí": label,
            property_id_1: a[col],
            property_id_2: b[col],
        })
    return pd.DataFrame(rows)

if len(recs) >= 2:
    comparison = compare_properties(recs.loc[0, "property_id"], recs.loc[1, "property_id"])
    display(comparison)


In [ ]:
# Cell 17 - Lưu dataset symbolic, rules, triples, matrix, kết quả mẫu
# Lưu ý: Không có cột source/source_name/source_file/source_sheet trong output.

EXPORT_COLUMNS = [
    "property_id",
    "title", "description", "url",
    "raw_location", "ward", "district", "location_zone", "property_type",
    "raw_price", "price_billion", "price_band",
    "raw_area", "area_m2", "area_band", "price_per_m2_million", "district_median_price_per_m2", "price_reasonable_score",
    "bedrooms", "bathrooms", "floors", "bedrooms_band",
    "frontage_m", "frontage_band", "road_width_m", "road_band",
    "legal_class", "legal_score",
    "amenity_tags",
    "family_suitability_score", "family_suitability_level",
    "business_potential_score", "business_potential_level",
    "rental_potential_score", "rental_potential_level",
    "investment_potential_score", "investment_potential_level",
    "risk_score", "risk_level",
    "data_quality_score", "data_quality_level", "data_flags",
    "symbolic_facts", "risk_flags", "triggered_rules", "rule_evidence",
]

# Chỉ lấy cột tồn tại
EXPORT_COLUMNS = [c for c in EXPORT_COLUMNS if c in bds.columns]

bds_export = bds[EXPORT_COLUMNS].copy()

# List/dict -> JSON string để CSV/XLSX dễ đọc
for col in ["amenity_tags", "data_flags", "symbolic_facts", "risk_flags", "triggered_rules", "rule_evidence"]:
    if col in bds_export.columns:
        bds_export[col] = bds_export[col].apply(lambda x: json.dumps(x, ensure_ascii=False))

bds_export.to_csv(OUTPUT_DIR / "bds_clean_symbolic_v2.csv", index=False, encoding="utf-8-sig")
bds_export.to_excel(OUTPUT_DIR / "bds_clean_symbolic_v2.xlsx", index=False)

with open(OUTPUT_DIR / "symbolic_rule_base_v2.json", "w", encoding="utf-8") as f:
    json.dump(RULE_BASE, f, ensure_ascii=False, indent=2)

kg_triples.to_csv(OUTPUT_DIR / "bds_kg_triples_v2.csv", index=False, encoding="utf-8-sig")
location_similarity_matrix.to_csv(OUTPUT_DIR / "location_similarity_matrix_v2.csv", encoding="utf-8-sig")

if 'recs' in globals() and len(recs) > 0:
    recs_export = recs.copy()
    for col in ["amenity_tags", "symbolic_facts", "risk_flags", "triggered_rules"]:
        if col in recs_export.columns:
            recs_export[col] = recs_export[col].apply(lambda x: json.dumps(x, ensure_ascii=False))
    recs_export.to_csv(OUTPUT_DIR / "sample_recommendation_results_v2.csv", index=False, encoding="utf-8-sig")
    recs_export.to_excel(OUTPUT_DIR / "sample_recommendation_results_v2.xlsx", index=False)

print("Đã lưu output:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print("-", p)


In [ ]:
# Cell 18 - Nén output để tải về trong Colab
import shutil

zip_path = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=str(OUTPUT_DIR))
print("File zip output:", zip_path)

# Nếu chạy trên Colab, có thể tải bằng:
# from google.colab import files
# files.download(zip_path)


# Bước 1, 2, 3 sau khi đã có dữ liệu Symbolic

Phần này thực hiện tiếp 3 việc:

1. **Kiểm tra chất lượng dữ liệu sau chuẩn hóa**: missing values, outlier, phân bố symbolic, chất lượng dữ liệu.
2. **Chốt và áp dụng tập Symbolic Facts + Rule Base chặt hơn**: tạo fact chuẩn, risk flag chuẩn, evidence rõ.
3. **Hoàn thiện Recommendation Scoring v3**: parse nhu cầu, hard filter có nới lỏng, fuzzy scoring, giải thích kết quả.


In [ ]:

# ================================
# STEP 1 - KIỂM TRA CHẤT LƯỢNG DỮ LIỆU SAU CHUẨN HÓA
# ================================

import os
import json
import ast
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

# Lấy dữ liệu từ notebook trước đó
if "bds" in globals():
    bds_step = bds.copy()
    print("Đã dùng biến bds hiện có.")
elif "bds_export" in globals():
    bds_step = bds_export.copy()
    print("Đã dùng biến bds_export hiện có.")
else:
    possible_paths = [
        "/content/bds_symbolic_output_v2/bds_clean_symbolic_v2.xlsx",
        "/content/bds_symbolic_output_v2/bds_clean_symbolic_v2.csv",
        "/content/bds_symbolic_output/bds_clean_symbolic_v2.xlsx",
        "/content/bds_symbolic_output/bds_clean_symbolic_v2.csv",
        "/content/bds_clean_symbolic_v2.xlsx",
        "/content/bds_clean_symbolic_v2.csv",
    ]
    found_path = None
    for p in possible_paths:
        if os.path.exists(p):
            found_path = p
            break

    if found_path is None:
        raise FileNotFoundError(
            "Không tìm thấy dữ liệu symbolic. Hãy chạy các cell chuẩn hóa trước."
        )

    if found_path.endswith(".xlsx"):
        bds_step = pd.read_excel(found_path)
    else:
        bds_step = pd.read_csv(found_path)

    print("Đã load dữ liệu từ:", found_path)

# Parse lại các cột list nếu đang là string do đọc từ CSV/Excel
def _safe_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, dict):
        return [x]
    if pd.isna(x):
        return []
    s = str(x).strip()
    if s == "" or s.lower() in ["nan", "none", "null"]:
        return []
    try:
        v = json.loads(s)
        if isinstance(v, list):
            return v
        if isinstance(v, dict):
            return [v]
        return [v]
    except Exception:
        pass
    try:
        v = ast.literal_eval(s)
        if isinstance(v, list):
            return v
        if isinstance(v, dict):
            return [v]
        return [v]
    except Exception:
        pass
    if "," in s:
        return [i.strip() for i in s.split(",") if i.strip()]
    return [s]

for c in ["amenity_tags", "data_flags", "symbolic_facts", "risk_flags", "triggered_rules", "rule_evidence"]:
    if c in bds_step.columns:
        bds_step[c] = bds_step[c].apply(_safe_list)

# Ép numeric cho các cột điểm/số
numeric_cols = [
    "price_billion", "area_m2", "price_per_m2_million",
    "bedrooms", "bathrooms", "floors", "frontage_m", "road_width_m",
    "legal_score", "price_reasonable_score",
    "family_suitability_score", "business_potential_score",
    "rental_potential_score", "investment_potential_score",
    "risk_score", "data_quality_score",
]
for c in numeric_cols:
    if c in bds_step.columns:
        bds_step[c] = pd.to_numeric(bds_step[c], errors="coerce")

# Các cột bắt buộc nên có
required_cols = [
    "property_id", "title", "raw_location", "district", "location_zone",
    "property_type", "price_billion", "area_m2", "price_per_m2_million",
    "legal_class", "legal_score", "amenity_tags", "data_flags",
    "family_suitability_score", "business_potential_score",
    "rental_potential_score", "investment_potential_score",
    "risk_score", "data_quality_score",
]
missing_required_cols = [c for c in required_cols if c not in bds_step.columns]

print("Kích thước dữ liệu:", bds_step.shape)
print("Các cột bắt buộc bị thiếu:", missing_required_cols)

# 1.1. Báo cáo thiếu dữ liệu
available_required = [c for c in required_cols if c in bds_step.columns]
missing_report = pd.DataFrame({
    "column": available_required,
    "missing_count": [bds_step[c].isna().sum() for c in available_required],
    "missing_ratio": [round(float(bds_step[c].isna().mean()), 4) for c in available_required],
})
missing_report["valid_count"] = len(bds_step) - missing_report["missing_count"]
missing_report = missing_report.sort_values("missing_ratio", ascending=False)

# 1.2. Báo cáo trùng lặp
duplicate_report = {
    "total_rows": int(len(bds_step)),
    "duplicated_property_id": int(bds_step["property_id"].duplicated().sum()) if "property_id" in bds_step.columns else None,
    "duplicated_url_non_empty": int(
        bds_step.loc[bds_step["url"].fillna("").astype(str).str.strip() != "", "url"].duplicated().sum()
    ) if "url" in bds_step.columns else None,
    "duplicated_title_location_price_area": None,
}

if {"title", "raw_location", "price_billion", "area_m2"}.issubset(bds_step.columns):
    fp = (
        bds_step["title"].fillna("").astype(str).str.lower().str[:80] + "|" +
        bds_step["raw_location"].fillna("").astype(str).str.lower().str[:80] + "|" +
        bds_step["price_billion"].round(2).astype(str) + "|" +
        bds_step["area_m2"].round(1).astype(str)
    )
    duplicate_report["duplicated_title_location_price_area"] = int(fp.duplicated().sum())

duplicate_report_df = pd.DataFrame([duplicate_report])

# 1.3. Outlier kiểm tra nhanh
def _outlier_count(series, low=None, high=None):
    s = pd.to_numeric(series, errors="coerce")
    mask = pd.Series(False, index=s.index)
    if low is not None:
        mask |= s < low
    if high is not None:
        mask |= s > high
    return int(mask.sum())

outlier_specs = {
    "price_billion": (0.1, 500),
    "area_m2": (10, 2000),
    "price_per_m2_million": (1, 1000),
    "bedrooms": (0, 30),
    "bathrooms": (0, 30),
    "floors": (0, 100),
    "frontage_m": (0.5, 100),
    "road_width_m": (0.5, 100),
}
outlier_rows = []
for col, (low, high) in outlier_specs.items():
    if col in bds_step.columns:
        outlier_rows.append({
            "column": col,
            "low_threshold": low,
            "high_threshold": high,
            "outlier_count": _outlier_count(bds_step[col], low, high),
            "valid_count": int(bds_step[col].notna().sum()),
            "median": round(float(bds_step[col].median()), 3) if bds_step[col].notna().sum() > 0 else np.nan,
            "mean": round(float(bds_step[col].mean()), 3) if bds_step[col].notna().sum() > 0 else np.nan,
        })
outlier_report = pd.DataFrame(outlier_rows)

# 1.4. Phân bố các biến symbolic
def value_count_df(col):
    if col not in bds_step.columns:
        return pd.DataFrame(columns=[col, "count", "ratio"])
    vc = bds_step[col].fillna("unknown").value_counts().reset_index()
    vc.columns = [col, "count"]
    vc["ratio"] = (vc["count"] / len(bds_step)).round(4)
    return vc

symbolic_distribution = {}
for col in [
    "district", "location_zone", "property_type", "price_band", "area_band",
    "bedrooms_band", "road_band", "frontage_band", "legal_class",
    "family_suitability_level", "business_potential_level",
    "rental_potential_level", "investment_potential_level",
    "risk_level", "data_quality_level",
]:
    if col in bds_step.columns:
        symbolic_distribution[col] = value_count_df(col)

# 1.5. Data flags, facts, risk flags
def explode_count(list_col, out_col_name):
    if list_col not in bds_step.columns:
        return pd.DataFrame(columns=[out_col_name, "count", "ratio"])
    rows = []
    for _, row in bds_step.iterrows():
        items = row[list_col]
        if not isinstance(items, list):
            items = []
        for item in items:
            rows.append(item)
    if len(rows) == 0:
        return pd.DataFrame(columns=[out_col_name, "count", "ratio"])
    out = pd.Series(rows).value_counts().reset_index()
    out.columns = [out_col_name, "count"]
    out["ratio"] = (out["count"] / len(bds_step)).round(4)
    return out

data_flag_count = explode_count("data_flags", "data_flag")
fact_count = explode_count("symbolic_facts", "symbolic_fact")
risk_flag_count = explode_count("risk_flags", "risk_flag")
rule_count = explode_count("triggered_rules", "triggered_rule")

# 1.6. KPI tổng quan
def _mean_col(col):
    if col in bds_step.columns and bds_step[col].notna().sum() > 0:
        return round(float(bds_step[col].mean()), 3)
    return np.nan

quality_kpis = {
    "total_properties": int(len(bds_step)),
    "valid_price_ratio": round(float(bds_step["price_billion"].notna().mean()), 4) if "price_billion" in bds_step.columns else np.nan,
    "valid_area_ratio": round(float(bds_step["area_m2"].notna().mean()), 4) if "area_m2" in bds_step.columns else np.nan,
    "valid_district_ratio": round(float((bds_step["district"].notna() & (bds_step["district"] != "Không rõ")).mean()), 4) if "district" in bds_step.columns else np.nan,
    "clear_legal_ratio": round(float((bds_step["legal_class"] == "clear_ownership").mean()), 4) if "legal_class" in bds_step.columns else np.nan,
    "avg_data_quality_score": _mean_col("data_quality_score"),
    "avg_risk_score": _mean_col("risk_score"),
    "avg_family_score": _mean_col("family_suitability_score"),
    "avg_business_score": _mean_col("business_potential_score"),
    "avg_rental_score": _mean_col("rental_potential_score"),
    "avg_investment_score": _mean_col("investment_potential_score"),
}
quality_kpis_df = pd.DataFrame([quality_kpis])

display(HTML("<h3>STEP 1. KPI chất lượng dữ liệu</h3>"))
display(quality_kpis_df)

display(HTML("<h3>Thiếu dữ liệu theo cột quan trọng</h3>"))
display(missing_report)

display(HTML("<h3>Kiểm tra trùng lặp</h3>"))
display(duplicate_report_df)

display(HTML("<h3>Outlier kiểm tra nhanh</h3>"))
display(outlier_report)

display(HTML("<h3>Top data flags</h3>"))
display(data_flag_count.head(30))

display(HTML("<h3>Top symbolic facts hiện tại</h3>"))
display(fact_count.head(30))

display(HTML("<h3>Top risk flags hiện tại</h3>"))
display(risk_flag_count.head(30))

# Lưu báo cáo chất lượng
QUALITY_OUTPUT_DIR = Path("/content/bds_step_1_2_3_output")
QUALITY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
quality_report_path = QUALITY_OUTPUT_DIR / "step1_data_quality_audit.xlsx"

with pd.ExcelWriter(quality_report_path, engine="openpyxl") as writer:
    quality_kpis_df.to_excel(writer, sheet_name="kpi", index=False)
    missing_report.to_excel(writer, sheet_name="missing_report", index=False)
    duplicate_report_df.to_excel(writer, sheet_name="duplicate_report", index=False)
    outlier_report.to_excel(writer, sheet_name="outlier_report", index=False)
    data_flag_count.to_excel(writer, sheet_name="data_flags", index=False)
    fact_count.to_excel(writer, sheet_name="symbolic_facts_old", index=False)
    risk_flag_count.to_excel(writer, sheet_name="risk_flags_old", index=False)
    rule_count.to_excel(writer, sheet_name="triggered_rules_old", index=False)

    for col, table in symbolic_distribution.items():
        safe_sheet = col[:31]
        table.to_excel(writer, sheet_name=safe_sheet, index=False)

print("Đã lưu báo cáo Step 1 tại:", quality_report_path)


In [ ]:

# ================================
# STEP 2 - CHỐT SYMBOLIC FACTS + RULE BASE CHẶT HƠN
# ================================

# Tập facts/risk flags được kiểm soát, dùng thống nhất trong báo cáo và hệ thống
SYMBOLIC_FACT_CATALOG_V3 = {
    "low_legal_risk": "Pháp lý rõ, rủi ro pháp lý thấp.",
    "high_data_confidence": "Dữ liệu đủ trường quan trọng, độ tin cậy cao.",
    "good_value_by_area": "Giá/m2 hợp lý so với trung vị khu vực.",
    "suitable_for_family": "Phù hợp nhu cầu mua để ở/gia đình.",
    "suitable_for_business": "Phù hợp kinh doanh nhờ mặt tiền, đường lớn hoặc tín hiệu kinh doanh.",
    "suitable_for_rental": "Có tiềm năng khai thác cho thuê.",
    "suitable_for_investment": "Có tiềm năng đầu tư.",
    "good_car_access": "Đường vào phù hợp xe hơi hoặc đường rộng.",
    "central_location": "Nằm trong nhóm khu vực trung tâm.",
    "developing_location": "Nằm trong nhóm khu vực đang phát triển.",
}

RISK_FLAG_CATALOG_V3 = {
    "need_legal_verification": "Cần kiểm tra lại pháp lý trước khi ra quyết định.",
    "low_data_confidence": "Dữ liệu thiếu nhiều trường quan trọng.",
    "possibly_overpriced": "Giá/m2 có thể cao so với mặt bằng khu vực.",
    "missing_essential_price": "Thiếu giá.",
    "missing_essential_area": "Thiếu diện tích.",
    "missing_essential_location": "Thiếu hoặc không xác định được vị trí.",
    "small_alley_access": "Đường/hẻm nhỏ, có thể hạn chế xe hơi hoặc kinh doanh.",
    "unknown_property_type": "Không xác định rõ loại hình BĐS.",
}

STRICT_RULE_BASE_V3 = [
    {
        "rule_id": "R01_LOW_LEGAL_RISK",
        "type": "fact",
        "conclusion": "low_legal_risk",
        "condition": "legal_class == clear_ownership",
        "threshold": None,
        "description": "Pháp lý rõ khi legal_class là clear_ownership.",
    },
    {
        "rule_id": "R02_NEED_LEGAL_VERIFICATION",
        "type": "risk",
        "conclusion": "need_legal_verification",
        "condition": "legal_class in unknown/pending/contract_based/other",
        "threshold": None,
        "description": "Pháp lý không rõ hoặc chưa chắc chắn cần kiểm tra.",
    },
    {
        "rule_id": "R03_HIGH_DATA_CONFIDENCE",
        "type": "fact",
        "conclusion": "high_data_confidence",
        "condition": "data_quality_score >= 0.85",
        "threshold": 0.85,
        "description": "Dữ liệu tin đăng có chất lượng cao.",
    },
    {
        "rule_id": "R04_LOW_DATA_CONFIDENCE",
        "type": "risk",
        "conclusion": "low_data_confidence",
        "condition": "data_quality_score < 0.70",
        "threshold": 0.70,
        "description": "Dữ liệu thiếu nhiều thông tin quan trọng.",
    },
    {
        "rule_id": "R05_GOOD_VALUE_BY_AREA",
        "type": "fact",
        "conclusion": "good_value_by_area",
        "condition": "price_reasonable_score >= 0.70",
        "threshold": 0.70,
        "description": "Giá/m2 hợp lý so với trung vị khu vực.",
    },
    {
        "rule_id": "R06_POSSIBLY_OVERPRICED",
        "type": "risk",
        "conclusion": "possibly_overpriced",
        "condition": "price_reasonable_score < 0.40",
        "threshold": 0.40,
        "description": "Giá/m2 cao hơn đáng kể so với mặt bằng khu vực.",
    },
    {
        "rule_id": "R07_SUITABLE_FOR_FAMILY",
        "type": "fact",
        "conclusion": "suitable_for_family",
        "condition": "family_suitability_score >= 0.70 AND legal_score >= 0.60 AND risk_score <= 0.50",
        "threshold": 0.70,
        "description": "Phù hợp gia đình nếu điểm family cao, pháp lý không quá rủi ro và rủi ro tổng không cao.",
    },
    {
        "rule_id": "R08_SUITABLE_FOR_BUSINESS",
        "type": "fact",
        "conclusion": "suitable_for_business",
        "condition": "business_potential_score >= 0.65 AND road/frontage not too weak",
        "threshold": 0.65,
        "description": "Phù hợp kinh doanh nếu có điểm kinh doanh cao và tiếp cận đường/mặt tiền tương đối tốt.",
    },
    {
        "rule_id": "R09_SUITABLE_FOR_RENTAL",
        "type": "fact",
        "conclusion": "suitable_for_rental",
        "condition": "rental_potential_score >= 0.65 AND data_quality_score >= 0.60",
        "threshold": 0.65,
        "description": "Phù hợp cho thuê nếu điểm rental cao và dữ liệu đủ tin cậy.",
    },
    {
        "rule_id": "R10_SUITABLE_FOR_INVESTMENT",
        "type": "fact",
        "conclusion": "suitable_for_investment",
        "condition": "investment_potential_score >= 0.70 AND legal_score >= 0.50 AND price_reasonable_score >= 0.40",
        "threshold": 0.70,
        "description": "Phù hợp đầu tư nếu điểm investment cao, pháp lý không quá yếu và giá không quá bất hợp lý.",
    },
    {
        "rule_id": "R11_GOOD_CAR_ACCESS",
        "type": "fact",
        "conclusion": "good_car_access",
        "condition": "road_width_m >= 5 OR road_band in car_access/wide_road OR amenity_tags contains car_access",
        "threshold": None,
        "description": "Đường vào phù hợp xe hơi hoặc đường rộng.",
    },
    {
        "rule_id": "R12_SMALL_ALLEY_ACCESS",
        "type": "risk",
        "conclusion": "small_alley_access",
        "condition": "road_width_m < 3 OR road_band == small_alley",
        "threshold": None,
        "description": "Đường/hẻm nhỏ có thể hạn chế kinh doanh và xe hơi.",
    },
    {
        "rule_id": "R13_CENTRAL_LOCATION",
        "type": "fact",
        "conclusion": "central_location",
        "condition": "location_zone == central",
        "threshold": None,
        "description": "BĐS nằm trong khu vực trung tâm.",
    },
    {
        "rule_id": "R14_DEVELOPING_LOCATION",
        "type": "fact",
        "conclusion": "developing_location",
        "condition": "location_zone == developing",
        "threshold": None,
        "description": "BĐS nằm trong khu vực đang phát triển.",
    },
    {
        "rule_id": "R15_MISSING_ESSENTIALS",
        "type": "risk",
        "conclusion": "missing essential fields",
        "condition": "missing price/area/location/property_type",
        "threshold": None,
        "description": "Cảnh báo thiếu các trường thiết yếu.",
    },
]

def _row_list(row, col):
    if col not in row.index:
        return []
    val = row[col]
    return val if isinstance(val, list) else _safe_list(val)

def _has_tag_v3(row, tag):
    return tag in _row_list(row, "amenity_tags")

def _num(row, col, default=np.nan):
    if col not in row.index:
        return default
    return row[col]

def apply_strict_symbolic_rules_v3(row):
    facts = []
    risk_flags = []
    triggered_rules = []
    evidence = []

    def add_fact(rule_id, fact, strength, ev):
        if fact not in facts:
            facts.append(fact)
        triggered_rules.append(rule_id)
        evidence.append({
            "rule_id": rule_id,
            "type": "fact",
            "conclusion": fact,
            "strength": round(float(strength), 3),
            "evidence": ev,
        })

    def add_risk(rule_id, risk, strength, ev):
        if risk not in risk_flags:
            risk_flags.append(risk)
        triggered_rules.append(rule_id)
        evidence.append({
            "rule_id": rule_id,
            "type": "risk",
            "conclusion": risk,
            "strength": round(float(strength), 3),
            "evidence": ev,
        })

    legal_class = row.get("legal_class", "unknown")
    legal_score = _num(row, "legal_score", 0.2)
    quality = _num(row, "data_quality_score", 0.0)
    risk = _num(row, "risk_score", 0.0)
    price_reasonable = _num(row, "price_reasonable_score", 0.5)
    family = _num(row, "family_suitability_score", 0.0)
    business = _num(row, "business_potential_score", 0.0)
    rental = _num(row, "rental_potential_score", 0.0)
    investment = _num(row, "investment_potential_score", 0.0)
    road_width = _num(row, "road_width_m", np.nan)
    road_band = row.get("road_band", "unknown")
    frontage_m = _num(row, "frontage_m", np.nan)
    location_zone_value = row.get("location_zone", "unknown")
    property_type = row.get("property_type", "unknown")
    data_flags = _row_list(row, "data_flags")

    # R01/R02
    if legal_class == "clear_ownership":
        add_fact("R01_LOW_LEGAL_RISK", "low_legal_risk", 1.0, f"legal_class={legal_class}, legal_score={legal_score}")
    elif legal_class in ["unknown", "pending", "contract_based", "other"]:
        strength = 0.95 if legal_class == "unknown" else 0.75
        add_risk("R02_NEED_LEGAL_VERIFICATION", "need_legal_verification", strength, f"legal_class={legal_class}, legal_score={legal_score}")

    # R03/R04
    if not pd.isna(quality) and quality >= 0.85:
        add_fact("R03_HIGH_DATA_CONFIDENCE", "high_data_confidence", quality, f"data_quality_score={quality}")
    if pd.isna(quality) or quality < 0.70:
        add_risk("R04_LOW_DATA_CONFIDENCE", "low_data_confidence", 1 - (quality if not pd.isna(quality) else 0), f"data_quality_score={quality}")

    # R05/R06
    if not pd.isna(price_reasonable) and price_reasonable >= 0.70:
        add_fact("R05_GOOD_VALUE_BY_AREA", "good_value_by_area", price_reasonable, f"price_reasonable_score={price_reasonable}")
    if not pd.isna(price_reasonable) and price_reasonable < 0.40:
        add_risk("R06_POSSIBLY_OVERPRICED", "possibly_overpriced", 1 - price_reasonable, f"price_reasonable_score={price_reasonable}")

    # R07
    if family >= 0.70 and legal_score >= 0.60 and risk <= 0.50:
        strength = min(1.0, 0.55 * family + 0.25 * legal_score + 0.20 * (1 - risk))
        add_fact(
            "R07_SUITABLE_FOR_FAMILY",
            "suitable_for_family",
            strength,
            f"family_score={family}, legal_score={legal_score}, risk_score={risk}, bedrooms={row.get('bedrooms')}, area_m2={row.get('area_m2')}",
        )

    # R08
    road_ok = (
        (not pd.isna(road_width) and road_width >= 3) or
        road_band in ["motorbike_alley", "car_access", "wide_road", "unknown"]
    )
    frontage_ok = pd.isna(frontage_m) or frontage_m >= 3.5
    if business >= 0.65 and road_ok and frontage_ok:
        strength = min(1.0, 0.75 * business + 0.15 * (0.7 if road_ok else 0.2) + 0.10 * (0.7 if frontage_ok else 0.2))
        add_fact(
            "R08_SUITABLE_FOR_BUSINESS",
            "suitable_for_business",
            strength,
            f"business_score={business}, frontage_m={frontage_m}, road_width_m={road_width}, road_band={road_band}, tags={row.get('amenity_tags')}",
        )

    # R09
    if rental >= 0.65 and quality >= 0.60:
        strength = min(1.0, 0.75 * rental + 0.25 * quality)
        add_fact(
            "R09_SUITABLE_FOR_RENTAL",
            "suitable_for_rental",
            strength,
            f"rental_score={rental}, data_quality_score={quality}, zone={location_zone_value}, type={property_type}",
        )

    # R10
    if investment >= 0.70 and legal_score >= 0.50 and price_reasonable >= 0.40:
        strength = min(1.0, 0.65 * investment + 0.20 * legal_score + 0.15 * price_reasonable)
        add_fact(
            "R10_SUITABLE_FOR_INVESTMENT",
            "suitable_for_investment",
            strength,
            f"investment_score={investment}, legal_score={legal_score}, price_reasonable_score={price_reasonable}, zone={location_zone_value}",
        )

    # R11/R12
    if (
        (not pd.isna(road_width) and road_width >= 5) or
        road_band in ["car_access", "wide_road"] or
        _has_tag_v3(row, "car_access")
    ):
        strength = 1.0 if (not pd.isna(road_width) and road_width >= 8) or road_band == "wide_road" else 0.75
        add_fact("R11_GOOD_CAR_ACCESS", "good_car_access", strength, f"road_width_m={road_width}, road_band={road_band}, tags={row.get('amenity_tags')}")
    if (not pd.isna(road_width) and road_width < 3) or road_band == "small_alley":
        add_risk("R12_SMALL_ALLEY_ACCESS", "small_alley_access", 0.80, f"road_width_m={road_width}, road_band={road_band}")

    # R13/R14
    if location_zone_value == "central":
        add_fact("R13_CENTRAL_LOCATION", "central_location", 1.0, f"location_zone={location_zone_value}, district={row.get('district')}")
    if location_zone_value == "developing":
        add_fact("R14_DEVELOPING_LOCATION", "developing_location", 0.85, f"location_zone={location_zone_value}, district={row.get('district')}")

    # R15 missing essentials
    if "missing_price" in data_flags or pd.isna(row.get("price_billion", np.nan)):
        add_risk("R15_MISSING_PRICE", "missing_essential_price", 0.90, "price_billion is missing")
    if "missing_area" in data_flags or pd.isna(row.get("area_m2", np.nan)):
        add_risk("R15_MISSING_AREA", "missing_essential_area", 0.85, "area_m2 is missing")
    if "missing_district" in data_flags or row.get("district", "Không rõ") == "Không rõ":
        add_risk("R15_MISSING_LOCATION", "missing_essential_location", 0.85, f"district={row.get('district')}")
    if property_type == "unknown":
        add_risk("R15_UNKNOWN_PROPERTY_TYPE", "unknown_property_type", 0.50, "property_type=unknown")

    # Remove duplicated triggered rules while preserving evidence entries
    facts = sorted(set(facts))
    risk_flags = sorted(set(risk_flags))
    triggered_rules_unique = []
    for r in triggered_rules:
        if r not in triggered_rules_unique:
            triggered_rules_unique.append(r)

    return pd.Series({
        "symbolic_facts_v3": facts,
        "risk_flags_v3": risk_flags,
        "triggered_rules_v3": triggered_rules_unique,
        "rule_evidence_v3": evidence,
    })

rule_outputs_v3 = bds_step.apply(apply_strict_symbolic_rules_v3, axis=1)
bds_v3 = pd.concat([bds_step.drop(columns=[c for c in rule_outputs_v3.columns if c in bds_step.columns], errors="ignore"), rule_outputs_v3], axis=1)

# Tạo level gọn hơn cho báo cáo
def _score_level(x, low=0.4, mid=0.7):
    if pd.isna(x):
        return "unknown"
    if x >= mid:
        return "high"
    if x >= low:
        return "medium"
    return "low"

for col in [
    "family_suitability_score", "business_potential_score",
    "rental_potential_score", "investment_potential_score",
    "data_quality_score",
]:
    if col in bds_v3.columns:
        bds_v3[col.replace("_score", "_level_v3")] = bds_v3[col].apply(_score_level)

if "risk_score" in bds_v3.columns:
    bds_v3["risk_level_v3"] = bds_v3["risk_score"].apply(
        lambda x: "unknown" if pd.isna(x) else ("high" if x >= 0.6 else ("medium" if x >= 0.3 else "low"))
    )

# Thống kê rule/fact/risk mới
fact_count_v3 = explode_count("symbolic_facts_v3", "symbolic_fact_v3") if "explode_count" in globals() else pd.DataFrame()
risk_flag_count_v3 = explode_count("risk_flags_v3", "risk_flag_v3") if "explode_count" in globals() else pd.DataFrame()
rule_count_v3 = explode_count("triggered_rules_v3", "triggered_rule_v3") if "explode_count" in globals() else pd.DataFrame()

display(HTML("<h3>STEP 2. Symbolic Facts v3</h3>"))
display(pd.DataFrame([{"fact": k, "meaning": v} for k, v in SYMBOLIC_FACT_CATALOG_V3.items()]))

display(HTML("<h3>Risk Flags v3</h3>"))
display(pd.DataFrame([{"risk_flag": k, "meaning": v} for k, v in RISK_FLAG_CATALOG_V3.items()]))

display(HTML("<h3>Rule Base v3</h3>"))
display(pd.DataFrame(STRICT_RULE_BASE_V3))

display(HTML("<h3>Tần suất facts v3</h3>"))
display(fact_count_v3)

display(HTML("<h3>Tần suất risk flags v3</h3>"))
display(risk_flag_count_v3)

display(HTML("<h3>Tần suất triggered rules v3</h3>"))
display(rule_count_v3)

# Cập nhật bds để các bước sau dùng bản chặt hơn
bds = bds_v3.copy()

# Lưu Step 2
STEP_OUTPUT_DIR = Path("/content/bds_step_1_2_3_output")
STEP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

bds_v3.to_excel(STEP_OUTPUT_DIR / "step2_bds_symbolic_rules_v3.xlsx", index=False)
bds_v3.to_csv(STEP_OUTPUT_DIR / "step2_bds_symbolic_rules_v3.csv", index=False, encoding="utf-8-sig")

with open(STEP_OUTPUT_DIR / "step2_symbolic_fact_catalog_v3.json", "w", encoding="utf-8") as f:
    json.dump(SYMBOLIC_FACT_CATALOG_V3, f, ensure_ascii=False, indent=2)

with open(STEP_OUTPUT_DIR / "step2_risk_flag_catalog_v3.json", "w", encoding="utf-8") as f:
    json.dump(RISK_FLAG_CATALOG_V3, f, ensure_ascii=False, indent=2)

with open(STEP_OUTPUT_DIR / "step2_strict_rule_base_v3.json", "w", encoding="utf-8") as f:
    json.dump(STRICT_RULE_BASE_V3, f, ensure_ascii=False, indent=2)

print("Đã lưu output Step 2 tại:", STEP_OUTPUT_DIR)


In [ ]:

# ================================
# STEP 2B - CẬP NHẬT KG TRIPLES TỪ RULES V3
# ================================

def create_triples_v3(df):
    triples = []

    numeric_predicates = {
        "price_billion": "hasPriceBillion",
        "area_m2": "hasAreaM2",
        "price_per_m2_million": "hasPricePerM2Million",
        "bedrooms": "hasBedrooms",
        "bathrooms": "hasBathrooms",
        "floors": "hasFloors",
        "frontage_m": "hasFrontageM",
        "road_width_m": "hasRoadWidthM",
        "legal_score": "hasLegalScore",
        "price_reasonable_score": "hasPriceReasonableScore",
        "family_suitability_score": "hasFamilySuitabilityScore",
        "business_potential_score": "hasBusinessPotentialScore",
        "rental_potential_score": "hasRentalPotentialScore",
        "investment_potential_score": "hasInvestmentPotentialScore",
        "risk_score": "hasRiskScore",
        "data_quality_score": "hasDataQualityScore",
    }

    categorical_predicates = {
        "property_type": "hasPropertyType",
        "district": "locatedInDistrict",
        "ward": "locatedInWard",
        "location_zone": "locatedInZone",
        "price_band": "hasPriceBand",
        "area_band": "hasAreaBand",
        "bedrooms_band": "hasBedroomBand",
        "road_band": "hasRoadBand",
        "frontage_band": "hasFrontageBand",
        "legal_class": "hasLegalClass",
        "risk_level_v3": "hasRiskLevel",
    }

    for _, row in df.iterrows():
        s = row["property_id"]
        triples.append([s, "type", "Property"])

        for col, pred in categorical_predicates.items():
            if col in df.columns:
                val = row.get(col)
                if not pd.isna(val):
                    triples.append([s, pred, str(val)])

        for col, pred in numeric_predicates.items():
            if col in df.columns:
                val = row.get(col, np.nan)
                if not pd.isna(val):
                    triples.append([s, pred, float(val)])

        for tag in _safe_list(row.get("amenity_tags", [])):
            triples.append([s, "hasAmenityTag", str(tag)])

        for fact in _safe_list(row.get("symbolic_facts_v3", [])):
            triples.append([s, "inferredFactV3", str(fact)])

        for risk in _safe_list(row.get("risk_flags_v3", [])):
            triples.append([s, "hasRiskFlagV3", str(risk)])

        for rule in _safe_list(row.get("triggered_rules_v3", [])):
            triples.append([s, "triggeredRuleV3", str(rule)])

        # Link evidence node để giải thích được
        for idx, ev in enumerate(_safe_list(row.get("rule_evidence_v3", []))):
            if isinstance(ev, dict):
                ev_node = f"{s}_Evidence_{idx+1:02d}"
                triples.append([s, "hasRuleEvidence", ev_node])
                triples.append([ev_node, "type", "RuleEvidence"])
                triples.append([ev_node, "forRule", ev.get("rule_id", "")])
                triples.append([ev_node, "hasEvidenceText", ev.get("evidence", "")])
                triples.append([ev_node, "hasEvidenceStrength", float(ev.get("strength", 0))])

    return pd.DataFrame(triples, columns=["subject", "predicate", "object"])

kg_triples_v3 = create_triples_v3(bds_v3)

display(HTML("<h3>KG triples v3</h3>"))
print("Số triples v3:", len(kg_triples_v3))
display(kg_triples_v3.head(30))

kg_triples_v3.to_csv(STEP_OUTPUT_DIR / "step2_kg_triples_v3.csv", index=False, encoding="utf-8-sig")
kg_triples_v3.to_excel(STEP_OUTPUT_DIR / "step2_kg_triples_v3.xlsx", index=False)

print("Đã lưu KG v3 tại:", STEP_OUTPUT_DIR / "step2_kg_triples_v3.csv")


In [ ]:

# ================================
# STEP 3 - HOÀN THIỆN RECOMMENDATION SCORING V3
# ================================

import re
from typing import Any, Dict, List, Tuple

# Fallback nếu các hàm từ notebook cũ chưa có trong runtime
def _clean_query_text(x):
    return str(x).strip() if x is not None else ""

def _normalize_query_text(x):
    if "norm_text" in globals():
        return norm_text(x)
    import unicodedata
    text = str(x).lower().strip()
    text = text.replace("đ", "d")
    text = unicodedata.normalize("NFD", text)
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    return re.sub(r"\s+", " ", text)

def _vn_float_local(x):
    if "vn_float" in globals():
        return vn_float(x)
    try:
        return float(str(x).replace(",", "."))
    except Exception:
        return np.nan

def _extract_district_local(text):
    if "extract_district" in globals():
        return extract_district(text)
    q = _normalize_query_text(text)
    m = re.search(r"\b(?:quan|q\.?)\s*(\d{1,2})\b", q)
    if m:
        return f"Quận {int(m.group(1))}"
    for name in ["thu duc", "binh thanh", "phu nhuan", "go vap", "tan binh", "tan phu", "binh tan", "nha be", "binh chanh", "hoc mon", "cu chi"]:
        if name in q:
            return name.title()
    return "Không rõ"

def _district_similarity_local(a, b):
    if "district_similarity" in globals():
        return district_similarity(a, b)
    if a == b:
        return 1.0
    return 0.35

def parse_user_query_v3(query):
    q_raw = _clean_query_text(query)
    q = _normalize_query_text(query)

    profile = {
        "raw_query": q_raw,
        "budget_billion": None,
        "target_area_m2": None,
        "district": None,
        "bedrooms_min": None,
        "bathrooms_min": None,
        "legal_required": False,
        "desired_amenities": [],
        "preferred_property_type": None,
        "purpose": "general",
        "hard_constraints": [],
        "soft_constraints": [],
    }

    # Ngân sách: khoảng 8 tỷ, dưới 10 tỷ, tầm 5.5 tỉ
    m = re.search(r"(?:khoang|tam|duoi|toi da|max|ngan sach|gia)?\s*(\d+(?:[\.,]\d+)?)\s*(ty|ti|tỉ|tỷ)", q)
    if m:
        profile["budget_billion"] = _vn_float_local(m.group(1))
        profile["soft_constraints"].append("budget")

    # Diện tích
    m = re.search(r"(\d+(?:[\.,]\d+)?)\s*(m2|m²)", q)
    if m:
        profile["target_area_m2"] = _vn_float_local(m.group(1))
        profile["soft_constraints"].append("area")

    # Phòng ngủ/phòng tắm
    m = re.search(r"(\d+)\s*(pn|phong ngu|phòng ngủ)", q)
    if m:
        profile["bedrooms_min"] = int(m.group(1))
        profile["hard_constraints"].append("bedrooms_min")

    m = re.search(r"(\d+)\s*(wc|phong tam|phòng tắm|ve sinh|nha ve sinh)", q)
    if m:
        profile["bathrooms_min"] = int(m.group(1))
        profile["soft_constraints"].append("bathrooms_min")

    # Vị trí
    d = _extract_district_local(q_raw)
    if d != "Không rõ":
        profile["district"] = d
        profile["soft_constraints"].append("location")

    # Pháp lý
    if any(k in q for k in ["phap ly ro", "so hong", "so do", "so rieng", "shr", "an toan phap ly"]):
        profile["legal_required"] = True
        profile["hard_constraints"].append("legal_required")

    # Loại BĐS
    if any(k in q for k in ["can ho", "chung cu", "apartment"]):
        profile["preferred_property_type"] = "apartment"
    elif any(k in q for k in ["nha pho", "mat tien", "mat pho", "shophouse"]):
        profile["preferred_property_type"] = "townhouse"
    elif any(k in q for k in ["biet thu", "villa"]):
        profile["preferred_property_type"] = "villa"
    elif any(k in q for k in ["nha rieng", "nha hem", "hxh", "hem xe hoi"]):
        profile["preferred_property_type"] = "house"
    elif any(k in q for k in ["dat nen", "lo dat", "dat"]):
        profile["preferred_property_type"] = "land"

    if profile["preferred_property_type"] is not None:
        profile["soft_constraints"].append("property_type")

    # Mục đích
    if any(k in q for k in ["kinh doanh", "mat tien", "mo cua hang", "van phong", "shop", "buon ban"]):
        profile["purpose"] = "business"
    elif any(k in q for k in ["dau tu", "tang gia", "sinh loi", "tiem nang", "luot song"]):
        profile["purpose"] = "investment"
    elif any(k in q for k in ["cho thue", "dong tien", "khai thac thue"]):
        profile["purpose"] = "rental"
    elif any(k in q for k in ["gia dinh", "cho con", "an cu", "de o", "o lau dai"]):
        profile["purpose"] = "family"

    # Tiện ích
    amenity_keywords = {
        "school": ["truong hoc", "truong", "cho con", "mam non", "dai hoc"],
        "hospital": ["benh vien", "phong kham", "y te"],
        "market": ["cho", "sieu thi", "bach hoa", "winmart", "coopmart"],
        "park": ["cong vien", "cay xanh"],
        "mall": ["trung tam thuong mai", "mall", "vincom", "lotte"],
        "business": ["kinh doanh", "shop", "van phong", "mat tien"],
        "rental": ["cho thue", "dong tien"],
        "car_access": ["oto", "xe hoi", "hem xe hoi", "duong rong"],
        "furnished": ["noi that", "full noi that"],
        "planning_infra": ["quy hoach", "ha tang", "metro", "cao toc"],
    }
    desired = []
    for tag, kws in amenity_keywords.items():
        if any(kw in q for kw in kws):
            desired.append(tag)
    profile["desired_amenities"] = sorted(set(desired))
    if desired:
        profile["soft_constraints"].append("amenity")

    return profile

PURPOSE_SCORE_COL_V3 = {
    "family": "family_suitability_score",
    "business": "business_potential_score",
    "rental": "rental_potential_score",
    "investment": "investment_potential_score",
    "general": None,
}

WEIGHTS_V3 = {
    "family": {
        "budget": 0.22,
        "price_reasonable": 0.08,
        "location": 0.20,
        "legal": 0.18,
        "area": 0.08,
        "amenity": 0.10,
        "type": 0.04,
        "purpose": 0.10,
    },
    "business": {
        "budget": 0.16,
        "price_reasonable": 0.07,
        "location": 0.20,
        "legal": 0.12,
        "area": 0.05,
        "amenity": 0.09,
        "type": 0.06,
        "purpose": 0.25,
    },
    "rental": {
        "budget": 0.16,
        "price_reasonable": 0.08,
        "location": 0.22,
        "legal": 0.12,
        "area": 0.05,
        "amenity": 0.09,
        "type": 0.06,
        "purpose": 0.22,
    },
    "investment": {
        "budget": 0.16,
        "price_reasonable": 0.18,
        "location": 0.18,
        "legal": 0.16,
        "area": 0.04,
        "amenity": 0.05,
        "type": 0.04,
        "purpose": 0.19,
    },
    "general": {
        "budget": 0.19,
        "price_reasonable": 0.10,
        "location": 0.19,
        "legal": 0.15,
        "area": 0.07,
        "amenity": 0.08,
        "type": 0.05,
        "purpose": 0.17,
    },
}

def budget_score_v3(price, budget):
    if budget is None or pd.isna(price):
        return 0.60
    if price <= budget:
        ratio = price / max(budget, 1e-9)
        if ratio >= 0.75:
            return 1.00
        if ratio >= 0.50:
            return 0.86
        return 0.70
    over = (price - budget) / max(budget, 1e-9)
    if over <= 0.05:
        return 0.90
    if over <= 0.10:
        return 0.78
    if over <= 0.20:
        return 0.55
    if over <= 0.35:
        return 0.30
    return 0.05

def area_score_v3(area, target):
    if target is None or pd.isna(area):
        return 0.60
    diff = abs(area - target) / max(target, 1)
    if diff <= 0.10:
        return 1.00
    if diff <= 0.25:
        return 0.80
    if diff <= 0.50:
        return 0.50
    return 0.20

def location_score_v3(row_district, desired_district):
    if desired_district is None:
        return 0.60
    return float(_district_similarity_local(row_district, desired_district))

def amenity_score_v3(row_tags, desired_tags):
    if not desired_tags:
        return 0.60
    tags = row_tags if isinstance(row_tags, list) else _safe_list(row_tags)
    if len(tags) == 0:
        return 0.0
    return len(set(tags) & set(desired_tags)) / max(len(set(desired_tags)), 1)

def property_type_score_v3(row_type, preferred_type):
    if preferred_type is None:
        return 0.60
    if row_type == preferred_type:
        return 1.00
    similar = {
        ("house", "townhouse"): 0.70,
        ("townhouse", "house"): 0.70,
        ("villa", "house"): 0.60,
        ("house", "villa"): 0.60,
        ("apartment", "house"): 0.35,
        ("house", "apartment"): 0.35,
        ("land", "house"): 0.30,
        ("land", "townhouse"): 0.30,
    }
    return similar.get((row_type, preferred_type), 0.25)

def purpose_score_v3(row, purpose):
    col = PURPOSE_SCORE_COL_V3.get(purpose)
    if col is None:
        vals = [
            row.get("family_suitability_score", 0),
            row.get("business_potential_score", 0),
            row.get("rental_potential_score", 0),
            row.get("investment_potential_score", 0),
        ]
        vals = [0 if pd.isna(v) else float(v) for v in vals]
        return max(vals)
    v = row.get(col, 0)
    return 0 if pd.isna(v) else float(v)

def hard_filter_v3(row, profile, relax_level=0):
    # legal_required là hard constraint. relax_level >= 2 cho phép legal_score >= 0.5 nhưng bị phạt trong scoring.
    if profile.get("legal_required"):
        if relax_level < 2 and row.get("legal_class") != "clear_ownership":
            return False
        if relax_level >= 2 and row.get("legal_score", 0) < 0.5:
            return False

    # bedroom_min
    bedrooms_min = profile.get("bedrooms_min")
    if bedrooms_min is not None:
        bedrooms = row.get("bedrooms", np.nan)
        if pd.isna(bedrooms):
            if relax_level == 0:
                return False
        else:
            required = bedrooms_min if relax_level == 0 else max(1, bedrooms_min - 1)
            if bedrooms < required:
                return False

    # bathrooms_min mềm hơn, chỉ loại ở relax_level 0
    bathrooms_min = profile.get("bathrooms_min")
    if bathrooms_min is not None and relax_level == 0:
        bathrooms = row.get("bathrooms", np.nan)
        if not pd.isna(bathrooms) and bathrooms < bathrooms_min:
            return False

    # Nếu ngân sách vượt quá nhiều, loại ở relax_level 0; relax_level 1/2 cho phép nhưng điểm thấp
    budget = profile.get("budget_billion")
    price = row.get("price_billion", np.nan)
    if budget is not None and not pd.isna(price):
        max_over = 1.25 if relax_level == 0 else (1.45 if relax_level == 1 else 1.80)
        if price > budget * max_over:
            return False

    return True

def component_scores_v3(row, profile):
    purpose = profile.get("purpose", "general")
    components = {
        "budget": budget_score_v3(row.get("price_billion", np.nan), profile.get("budget_billion")),
        "price_reasonable": 0.60 if pd.isna(row.get("price_reasonable_score", np.nan)) else float(row.get("price_reasonable_score")),
        "location": location_score_v3(row.get("district", "Không rõ"), profile.get("district")),
        "legal": float(row.get("legal_score", 0.2)) if not pd.isna(row.get("legal_score", np.nan)) else 0.2,
        "area": area_score_v3(row.get("area_m2", np.nan), profile.get("target_area_m2")),
        "amenity": amenity_score_v3(row.get("amenity_tags", []), profile.get("desired_amenities", [])),
        "type": property_type_score_v3(row.get("property_type", "unknown"), profile.get("preferred_property_type")),
        "purpose": purpose_score_v3(row, purpose),
    }

    # Phạt nếu user yêu cầu pháp lý rõ nhưng thực tế chỉ vừa đủ do relax
    if profile.get("legal_required") and row.get("legal_class") != "clear_ownership":
        components["legal"] *= 0.55

    return components

def final_score_v3(row, profile):
    purpose = profile.get("purpose", "general")
    weights = WEIGHTS_V3.get(purpose, WEIGHTS_V3["general"])
    comps = component_scores_v3(row, profile)

    base = sum(comps[k] * weights[k] for k in weights)

    quality = row.get("data_quality_score", 0.7)
    risk = row.get("risk_score", 0.0)
    if pd.isna(quality):
        quality = 0.5
    if pd.isna(risk):
        risk = 0.2

    quality_factor = 0.75 + 0.25 * float(quality)
    risk_factor = 1 - 0.35 * float(risk)

    adjusted = base * quality_factor * risk_factor

    return round(float(adjusted), 4), {k: round(float(v), 4) for k, v in comps.items()}, round(float(base), 4), round(float(quality_factor), 4), round(float(risk_factor), 4)

def explain_recommendation_v3(row, profile, comps, final_score):
    parts = []
    pid = row.get("property_id", "")
    parts.append(f"{pid} đạt điểm tổng {final_score:.3f}.")

    if profile.get("budget_billion") is not None:
        price = row.get("price_billion", np.nan)
        if not pd.isna(price):
            parts.append(f"Giá {price:.2f} tỷ so với ngân sách {profile['budget_billion']:.2f} tỷ, điểm ngân sách {comps['budget']:.2f}.")

    if profile.get("district") is not None:
        parts.append(f"Vị trí {row.get('district')} có điểm tương đồng {comps['location']:.2f} với {profile['district']}.")

    if row.get("legal_class") == "clear_ownership":
        parts.append("Pháp lý thuộc nhóm rõ ràng.")
    elif profile.get("legal_required"):
        parts.append("Pháp lý chưa đạt mức rõ hoàn toàn nên cần kiểm tra kỹ.")

    purpose = profile.get("purpose", "general")
    if purpose != "general":
        parts.append(f"Điểm phù hợp mục đích {purpose} là {comps['purpose']:.2f}.")

    facts = _safe_list(row.get("symbolic_facts_v3", []))
    risks = _safe_list(row.get("risk_flags_v3", []))

    if facts:
        parts.append("Facts suy luận: " + ", ".join(facts[:5]) + ".")
    if risks:
        parts.append("Cảnh báo: " + ", ".join(risks[:5]) + ".")

    # Lấy bằng chứng luật mạnh nhất
    evidences = _safe_list(row.get("rule_evidence_v3", []))
    if evidences:
        evidences_sorted = sorted(
            [e for e in evidences if isinstance(e, dict)],
            key=lambda x: float(x.get("strength", 0)),
            reverse=True
        )
        top_evs = evidences_sorted[:2]
        ev_text = []
        for ev in top_evs:
            ev_text.append(f"{ev.get('rule_id')}: {ev.get('evidence')}")
        if ev_text:
            parts.append("Bằng chứng luật: " + " | ".join(ev_text) + ".")

    return " ".join(parts)

def recommend_v3(query, df=None, top_k=10, relax=True):
    if df is None:
        df = bds_v3 if "bds_v3" in globals() else bds

    profile = parse_user_query_v3(query)

    selected = []
    relax_used = None

    relax_levels = [0, 1, 2] if relax else [0]
    for level in relax_levels:
        rows = []
        for _, row in df.iterrows():
            if hard_filter_v3(row, profile, relax_level=level):
                score, comps, base, qf, rf = final_score_v3(row, profile)
                rows.append({
                    "property_id": row.get("property_id"),
                    "title": row.get("title"),
                    "district": row.get("district"),
                    "location_zone": row.get("location_zone"),
                    "property_type": row.get("property_type"),
                    "price_billion": row.get("price_billion"),
                    "area_m2": row.get("area_m2"),
                    "bedrooms": row.get("bedrooms"),
                    "bathrooms": row.get("bathrooms"),
                    "legal_class": row.get("legal_class"),
                    "price_reasonable_score": row.get("price_reasonable_score"),
                    "risk_score": row.get("risk_score"),
                    "data_quality_score": row.get("data_quality_score"),
                    "final_score_v3": score,
                    "base_score_v3": base,
                    "quality_factor": qf,
                    "risk_factor": rf,
                    "budget_component": comps["budget"],
                    "location_component": comps["location"],
                    "legal_component": comps["legal"],
                    "area_component": comps["area"],
                    "amenity_component": comps["amenity"],
                    "type_component": comps["type"],
                    "purpose_component": comps["purpose"],
                    "symbolic_facts_v3": row.get("symbolic_facts_v3", []),
                    "risk_flags_v3": row.get("risk_flags_v3", []),
                    "triggered_rules_v3": row.get("triggered_rules_v3", []),
                    "explanation_v3": explain_recommendation_v3(row, profile, comps, score),
                    "url": row.get("url"),
                })
        if len(rows) > 0:
            selected = rows
            relax_used = level
            break

    rec_df = pd.DataFrame(selected)
    if len(rec_df) > 0:
        rec_df = rec_df.sort_values("final_score_v3", ascending=False).head(top_k).reset_index(drop=True)

    return profile, relax_used, rec_df

# Test nhanh với một số query mẫu
test_queries_v3 = [
    "Tìm nhà quanh quận 3 khoảng 8 tỷ, 2PN trở lên, pháp lý rõ, phù hợp gia đình",
    "Tìm nhà mặt tiền kinh doanh dưới 10 tỷ, đường xe hơi, pháp lý rõ",
    "Tìm căn hộ cho thuê ở Quận 7, khoảng 5 tỷ, có nội thất",
    "Tìm bất động sản đầu tư ở Thủ Đức, giá hợp lý, pháp lý rõ",
]

all_test_results = []
for q in test_queries_v3:
    profile, relax_used, recs = recommend_v3(q, top_k=5)
    print("=" * 120)
    print("QUERY:", q)
    print("PROFILE:", json.dumps(profile, ensure_ascii=False, indent=2))
    print("RELAX_LEVEL_USED:", relax_used)
    display(recs[[
        "property_id", "final_score_v3", "district", "property_type",
        "price_billion", "area_m2", "bedrooms", "legal_class",
        "price_reasonable_score", "risk_score", "data_quality_score",
        "symbolic_facts_v3", "risk_flags_v3"
    ]] if len(recs) > 0 else recs)

    for _, r in recs.iterrows():
        all_test_results.append({
            "query": q,
            "relax_level_used": relax_used,
            **r.to_dict()
        })

test_results_v3 = pd.DataFrame(all_test_results)

# Lưu Step 3
STEP_OUTPUT_DIR = Path("/content/bds_step_1_2_3_output")
STEP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

test_results_v3.to_excel(STEP_OUTPUT_DIR / "step3_recommendation_test_results_v3.xlsx", index=False)
test_results_v3.to_csv(STEP_OUTPUT_DIR / "step3_recommendation_test_results_v3.csv", index=False, encoding="utf-8-sig")

# Lưu profile test để đưa vào báo cáo
with open(STEP_OUTPUT_DIR / "step3_test_queries_v3.json", "w", encoding="utf-8") as f:
    json.dump(test_queries_v3, f, ensure_ascii=False, indent=2)

print("Đã lưu output Step 3 tại:", STEP_OUTPUT_DIR)


In [ ]:

# ================================
# STEP 3B - DÙNG THỬ RECOMMENDER V3 VỚI CÂU HỎI CỦA BẠN
# ================================

query_input = "Tìm nhà quanh quận 3 khoảng 8 tỷ, 2PN trở lên, pháp lý rõ, phù hợp gia đình"

profile, relax_used, recs = recommend_v3(query_input, top_k=10)

print("QUERY:")
print(query_input)

print("\nSYMBOLIC PROFILE:")
print(json.dumps(profile, ensure_ascii=False, indent=2))

print("\nRELAX_LEVEL_USED:", relax_used)

display(recs[[
    "property_id", "final_score_v3", "district", "location_zone", "property_type",
    "price_billion", "area_m2", "bedrooms", "bathrooms", "legal_class",
    "budget_component", "location_component", "legal_component",
    "purpose_component",
    "price_reasonable_score", "risk_score", "data_quality_score",
    "symbolic_facts_v3", "risk_flags_v3"
]])

print("\nGIẢI THÍCH TOP KẾT QUẢ:")
for i, row in recs.head(5).iterrows():
    print("=" * 120)
    print(f"TOP {i+1}: {row['property_id']} | Score={row['final_score_v3']:.3f}")
    print(row["explanation_v3"])
    if isinstance(row.get("url"), str) and row["url"].strip():
        print("URL:", row["url"])


In [ ]:

# ================================
# NÉN TOÀN BỘ OUTPUT STEP 1,2,3 ĐỂ TẢI VỀ
# ================================

import shutil

STEP_OUTPUT_DIR = Path("/content/bds_step_1_2_3_output")
zip_path = shutil.make_archive(str(STEP_OUTPUT_DIR), "zip", root_dir=str(STEP_OUTPUT_DIR))

print("File zip output Step 1,2,3:")
print(zip_path)

# Nếu chạy trên Colab, mở 2 dòng dưới:
# from google.colab import files
# files.download(zip_path)



# Bước 4, 5, 6: Explanation, Test Queries, Baseline Evaluation

Phần này tiếp tục sau Step 1, 2, 3:

4. **Module giải thích kết quả**: biến điểm số, facts, risk flags và rule evidence thành giải thích rõ ràng cho từng BĐS.
5. **Tạo bộ test queries**: xây tập câu hỏi mẫu theo các mục đích mua để ở, kinh doanh, cho thuê, đầu tư và nhu cầu tổng quát.
6. **So sánh baseline**: so sánh phương pháp đề xuất với Keyword Search, Hard Filter và TF-IDF Search bằng Precision@K, HitRate@K, MRR@K và NDCG@K.


In [ ]:

# ================================
# STEP 4 - MODULE GIẢI THÍCH KẾT QUẢ KHUYẾN NGHỊ
# ================================

import os
import json
import ast
import math
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

STEP456_OUTPUT_DIR = Path("/content/bds_step_4_5_6_output")
STEP456_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Lấy dữ liệu đã qua Step 1,2,3.
# Ưu tiên bds_v3/bds trong runtime; nếu chưa có thì load file Step 2 đã lưu.
if "bds_v3" in globals():
    bds_s456 = bds_v3.copy()
    print("Đã dùng biến bds_v3 trong notebook.")
elif "bds" in globals():
    bds_s456 = bds.copy()
    print("Đã dùng biến bds trong notebook.")
else:
    possible_paths = [
        "/content/bds_step_1_2_3_output/step2_bds_symbolic_rules_v3.xlsx",
        "/content/bds_step_1_2_3_output/step2_bds_symbolic_rules_v3.csv",
        "/content/bds_symbolic_output_v2/bds_clean_symbolic_v2.xlsx",
        "/content/bds_symbolic_output_v2/bds_clean_symbolic_v2.csv",
    ]
    found_path = None
    for p in possible_paths:
        if os.path.exists(p):
            found_path = p
            break
    if found_path is None:
        raise FileNotFoundError("Không tìm thấy dữ liệu Step 1,2,3. Hãy chạy notebook từ đầu đến Step 3 trước.")
    if found_path.endswith(".xlsx"):
        bds_s456 = pd.read_excel(found_path)
    else:
        bds_s456 = pd.read_csv(found_path)
    print("Đã load dữ liệu từ:", found_path)

# Fallback parse list/dict nếu đọc từ Excel/CSV làm list thành string.
def _safe_list_456(x):
    if isinstance(x, list):
        return x
    if isinstance(x, dict):
        return [x]
    if pd.isna(x):
        return []
    s = str(x).strip()
    if s == "" or s.lower() in ["nan", "none", "null"]:
        return []
    try:
        v = json.loads(s)
        if isinstance(v, list):
            return v
        if isinstance(v, dict):
            return [v]
        return [v]
    except Exception:
        pass
    try:
        v = ast.literal_eval(s)
        if isinstance(v, list):
            return v
        if isinstance(v, dict):
            return [v]
        return [v]
    except Exception:
        pass
    if "," in s:
        return [i.strip() for i in s.split(",") if i.strip()]
    return [s]

for c in [
    "amenity_tags", "data_flags", "symbolic_facts", "risk_flags", "triggered_rules", "rule_evidence",
    "symbolic_facts_v3", "risk_flags_v3", "triggered_rules_v3", "rule_evidence_v3",
]:
    if c in bds_s456.columns:
        bds_s456[c] = bds_s456[c].apply(_safe_list_456)

# Ép numeric các cột hay dùng.
for c in [
    "price_billion", "area_m2", "price_per_m2_million",
    "bedrooms", "bathrooms", "floors", "frontage_m", "road_width_m",
    "legal_score", "price_reasonable_score", "family_suitability_score",
    "business_potential_score", "rental_potential_score", "investment_potential_score",
    "risk_score", "data_quality_score",
]:
    if c in bds_s456.columns:
        bds_s456[c] = pd.to_numeric(bds_s456[c], errors="coerce")

# Nếu recommend_v3 chưa có vì chưa chạy Step 3 thì báo rõ.
if "recommend_v3" not in globals() or "parse_user_query_v3" not in globals():
    raise RuntimeError("Chưa tìm thấy recommend_v3/parse_user_query_v3. Hãy chạy Step 3 trước khi chạy Step 4.")

# Catalog tiếng Việt để giải thích dễ hiểu.
FACT_EXPLANATION_VI = {
    "low_legal_risk": "pháp lý rõ, rủi ro pháp lý thấp",
    "high_data_confidence": "dữ liệu tin đăng tương đối đầy đủ",
    "good_value_by_area": "giá/m² hợp lý so với mặt bằng khu vực",
    "suitable_for_family": "phù hợp nhu cầu mua để ở hoặc gia đình",
    "suitable_for_business": "có tín hiệu phù hợp kinh doanh",
    "suitable_for_rental": "có tiềm năng khai thác cho thuê",
    "suitable_for_investment": "có tiềm năng đầu tư",
    "good_car_access": "đường vào phù hợp xe hơi hoặc tiếp cận tốt",
    "central_location": "nằm trong khu vực trung tâm",
    "developing_location": "nằm trong khu vực đang phát triển",
}

RISK_EXPLANATION_VI = {
    "need_legal_verification": "cần kiểm tra lại pháp lý",
    "low_data_confidence": "dữ liệu thiếu một số thông tin quan trọng",
    "possibly_overpriced": "giá/m² có thể cao so với mặt bằng khu vực",
    "missing_essential_price": "thiếu thông tin giá",
    "missing_essential_area": "thiếu thông tin diện tích",
    "missing_essential_location": "thiếu hoặc chưa xác định được vị trí",
    "small_alley_access": "đường/hẻm nhỏ, có thể hạn chế xe hơi hoặc kinh doanh",
    "unknown_property_type": "chưa xác định rõ loại hình BĐS",
}

PURPOSE_VI = {
    "family": "mua để ở/gia đình",
    "business": "kinh doanh",
    "rental": "cho thuê",
    "investment": "đầu tư",
    "general": "nhu cầu tổng quát",
}

def _fmt_float(x, ndigits=2, suffix=""):
    if x is None or pd.isna(x):
        return "không rõ"
    return f"{float(x):.{ndigits}f}{suffix}"

def _get_original_row_by_id(property_id, df=bds_s456):
    sub = df[df["property_id"] == property_id]
    if len(sub) == 0:
        return None
    return sub.iloc[0]

def _top_rule_evidence(row, n=3):
    evidences = _safe_list_456(row.get("rule_evidence_v3", row.get("rule_evidence", [])))
    evs = [e for e in evidences if isinstance(e, dict)]
    evs = sorted(evs, key=lambda x: float(x.get("strength", 0)), reverse=True)
    return evs[:n]

def build_explanation_v4(original_row, rec_row, profile):
    """
    Tạo giải thích có cấu trúc cho một BĐS.
    Input:
      - original_row: dòng gốc trong bds_s456, chứa facts/rules/evidence.
      - rec_row: dòng trong kết quả recommend_v3, chứa component scores.
      - profile: symbolic profile parse từ query.
    Output: dict các trường giải thích để hiển thị/báo cáo.
    """
    if original_row is None:
        original_row = rec_row

    pid = rec_row.get("property_id", original_row.get("property_id", ""))
    purpose = profile.get("purpose", "general")
    purpose_text = PURPOSE_VI.get(purpose, purpose)

    facts = _safe_list_456(original_row.get("symbolic_facts_v3", rec_row.get("symbolic_facts_v3", [])))
    risks = _safe_list_456(original_row.get("risk_flags_v3", rec_row.get("risk_flags_v3", [])))
    rules = _safe_list_456(original_row.get("triggered_rules_v3", rec_row.get("triggered_rules_v3", [])))
    tags = _safe_list_456(original_row.get("amenity_tags", []))

    final_score = rec_row.get("final_score_v3", np.nan)
    budget_score = rec_row.get("budget_component", np.nan)
    location_score = rec_row.get("location_component", np.nan)
    legal_component = rec_row.get("legal_component", np.nan)
    purpose_component = rec_row.get("purpose_component", np.nan)
    risk_factor = rec_row.get("risk_factor", np.nan)
    quality_factor = rec_row.get("quality_factor", np.nan)

    price = original_row.get("price_billion", rec_row.get("price_billion", np.nan))
    area = original_row.get("area_m2", rec_row.get("area_m2", np.nan))
    district = original_row.get("district", rec_row.get("district", "Không rõ"))
    property_type = original_row.get("property_type", rec_row.get("property_type", "unknown"))
    legal_class = original_row.get("legal_class", rec_row.get("legal_class", "unknown"))

    positive_points = []
    if not pd.isna(location_score) and location_score >= 0.75:
        positive_points.append(f"Vị trí {district} phù hợp/tương đồng cao với khu vực mong muốn, điểm vị trí {_fmt_float(location_score)}.")
    elif profile.get("district") is not None:
        positive_points.append(f"Vị trí {district} chỉ tương đồng một phần với {profile.get('district')}, điểm vị trí {_fmt_float(location_score)}.")

    if profile.get("budget_billion") is not None:
        positive_points.append(
            f"Giá {_fmt_float(price, 2, ' tỷ')} so với ngân sách {_fmt_float(profile.get('budget_billion'), 2, ' tỷ')}, điểm ngân sách {_fmt_float(budget_score)}."
        )

    if legal_class == "clear_ownership":
        positive_points.append(f"Pháp lý thuộc nhóm clear_ownership, điểm pháp lý {_fmt_float(legal_component)}.")

    if purpose != "general":
        positive_points.append(f"Điểm phù hợp mục đích {purpose_text} là {_fmt_float(purpose_component)}.")

    readable_facts = [FACT_EXPLANATION_VI.get(f, f) for f in facts]
    readable_risks = [RISK_EXPLANATION_VI.get(r, r) for r in risks]

    if readable_facts:
        positive_points.append("Facts suy luận: " + "; ".join(readable_facts[:6]) + ".")

    warnings = []
    if readable_risks:
        warnings.append("Cảnh báo: " + "; ".join(readable_risks[:6]) + ".")
    if not pd.isna(risk_factor) and risk_factor < 0.90:
        warnings.append(f"Điểm bị phạt bởi rủi ro, risk_factor={_fmt_float(risk_factor)}.")
    if not pd.isna(quality_factor) and quality_factor < 0.92:
        warnings.append(f"Độ đầy đủ dữ liệu chưa cao, quality_factor={_fmt_float(quality_factor)}.")

    evidence_lines = []
    for ev in _top_rule_evidence(original_row, n=3):
        evidence_lines.append(
            f"{ev.get('rule_id')}: {ev.get('conclusion')} | strength={ev.get('strength')} | {ev.get('evidence')}"
        )

    if len(warnings) == 0:
        decision = f"{pid} là lựa chọn khá tốt cho mục tiêu {purpose_text} vì có nhiều facts tích cực và ít cảnh báo rủi ro."
    else:
        decision = f"{pid} có thể được cân nhắc cho mục tiêu {purpose_text}, nhưng cần kiểm tra các cảnh báo trước khi ra quyết định."

    short_explanation = " ".join([decision] + positive_points[:3] + warnings[:1])
    detailed_explanation = "\n".join(
        [
            f"BĐS: {pid}",
            f"Điểm tổng: {_fmt_float(final_score, 3)}",
            f"Loại: {property_type} | Khu vực: {district} | Giá: {_fmt_float(price, 2, ' tỷ')} | Diện tích: {_fmt_float(area, 1, ' m²')} | Pháp lý: {legal_class}",
            "",
            "Điểm mạnh:",
            *[f"- {p}" for p in positive_points],
            "",
            "Cảnh báo:",
            *([f"- {w}" for w in warnings] if warnings else ["- Chưa phát hiện cảnh báo lớn từ tập luật hiện tại."]),
            "",
            "Bằng chứng luật:",
            *([f"- {e}" for e in evidence_lines] if evidence_lines else ["- Không có rule evidence chi tiết."]),
            "",
            f"Triggered rules: {', '.join(rules[:8]) if rules else 'Không có'}",
            f"Amenity tags: {', '.join(tags[:8]) if tags else 'Không có'}",
        ]
    )

    return {
        "decision_summary_v4": decision,
        "short_explanation_v4": short_explanation,
        "detailed_explanation_v4": detailed_explanation,
        "positive_points_v4": positive_points,
        "warnings_v4": warnings,
        "rule_evidence_text_v4": evidence_lines,
    }

def recommend_v4(query, df=None, top_k=10, relax=True):
    """
    Wrapper quanh recommend_v3, bổ sung giải thích v4.
    """
    if df is None:
        df = bds_s456

    profile, relax_used, recs = recommend_v3(query, df=df, top_k=top_k, relax=relax)
    if len(recs) == 0:
        return profile, relax_used, recs

    explanation_rows = []
    for _, rec_row in recs.iterrows():
        original_row = _get_original_row_by_id(rec_row["property_id"], df=df)
        exp = build_explanation_v4(original_row, rec_row, profile)
        explanation_rows.append(exp)

    exp_df = pd.DataFrame(explanation_rows)
    recs_v4 = pd.concat([recs.reset_index(drop=True), exp_df.reset_index(drop=True)], axis=1)
    return profile, relax_used, recs_v4

# Test Step 4
sample_query_v4 = "Tìm nhà quanh quận 3 khoảng 8 tỷ, 2PN trở lên, pháp lý rõ, phù hợp gia đình"
profile_v4, relax_used_v4, recs_v4 = recommend_v4(sample_query_v4, top_k=5)

print("QUERY:", sample_query_v4)
print("PROFILE:", json.dumps(profile_v4, ensure_ascii=False, indent=2))
print("RELAX_LEVEL_USED:", relax_used_v4)

display(recs_v4[[
    "property_id", "final_score_v3", "district", "property_type", "price_billion", "area_m2",
    "legal_class", "risk_score", "data_quality_score", "decision_summary_v4", "short_explanation_v4"
]])

print("\nGIẢI THÍCH CHI TIẾT TOP 1:\n")
if len(recs_v4) > 0:
    print(recs_v4.loc[0, "detailed_explanation_v4"])

# Lưu output Step 4
recs_v4.to_excel(STEP456_OUTPUT_DIR / "step4_explanation_sample_v4.xlsx", index=False)
recs_v4.to_csv(STEP456_OUTPUT_DIR / "step4_explanation_sample_v4.csv", index=False, encoding="utf-8-sig")
print("Đã lưu Step 4 tại:", STEP456_OUTPUT_DIR)



## Giải thích Step 4

Step 4 không thay đổi điểm xếp hạng. Nó lấy kết quả từ `recommend_v3`, sau đó đọc lại `symbolic_facts_v3`, `risk_flags_v3`, `triggered_rules_v3` và `rule_evidence_v3` để tạo giải thích. Vì vậy, phần giải thích bám trực tiếp vào luật và bằng chứng, không phải mô tả chung chung.

Kết quả chính của Step 4 là các cột:

- `decision_summary_v4`: kết luận ngắn gọn.
- `short_explanation_v4`: lời giải thích ngắn để hiển thị trong app.
- `detailed_explanation_v4`: lời giải thích chi tiết cho báo cáo/demo.
- `positive_points_v4`: các điểm mạnh.
- `warnings_v4`: các cảnh báo.
- `rule_evidence_text_v4`: bằng chứng luật dùng để giải thích.


In [ ]:

# ================================
# STEP 5 - TẠO BỘ TEST QUERIES CHUẨN ĐỂ KIỂM THỬ HỆ THỐNG
# ================================

TEST_QUERY_SUITE_V4 = [
    {
        "case_id": "Q01_FAMILY_Q3_8B",
        "query": "Tìm nhà quanh quận 3 khoảng 8 tỷ, 2PN trở lên, pháp lý rõ, phù hợp gia đình",
        "purpose": "family",
        "expected_facts": ["suitable_for_family", "low_legal_risk"],
        "expected_risks_should_avoid": ["need_legal_verification", "low_data_confidence"],
    },
    {
        "case_id": "Q02_BUSINESS_10B",
        "query": "Tìm nhà mặt tiền kinh doanh dưới 10 tỷ, đường xe hơi, pháp lý rõ",
        "purpose": "business",
        "expected_facts": ["suitable_for_business", "good_car_access", "low_legal_risk"],
        "expected_risks_should_avoid": ["small_alley_access", "need_legal_verification"],
    },
    {
        "case_id": "Q03_RENTAL_Q7_5B",
        "query": "Tìm căn hộ cho thuê ở Quận 7 khoảng 5 tỷ, có nội thất, dữ liệu rõ ràng",
        "purpose": "rental",
        "expected_facts": ["suitable_for_rental", "high_data_confidence"],
        "expected_risks_should_avoid": ["low_data_confidence"],
    },
    {
        "case_id": "Q04_INVEST_THUDUC",
        "query": "Tìm bất động sản đầu tư ở Thủ Đức, giá hợp lý, pháp lý rõ, có tiềm năng hạ tầng",
        "purpose": "investment",
        "expected_facts": ["suitable_for_investment", "good_value_by_area", "low_legal_risk"],
        "expected_risks_should_avoid": ["possibly_overpriced", "need_legal_verification"],
    },
    {
        "case_id": "Q05_FAMILY_5B",
        "query": "Tìm nhà khoảng 5 tỷ để ở, 2 phòng ngủ trở lên, gần trường học và chợ",
        "purpose": "family",
        "expected_facts": ["suitable_for_family"],
        "expected_risks_should_avoid": ["low_data_confidence"],
    },
    {
        "case_id": "Q06_BUSINESS_CENTRAL",
        "query": "Cần nhà phố trung tâm để mở văn phòng kinh doanh, đường rộng, pháp lý rõ",
        "purpose": "business",
        "expected_facts": ["suitable_for_business", "central_location", "good_car_access"],
        "expected_risks_should_avoid": ["small_alley_access"],
    },
    {
        "case_id": "Q07_RENTAL_APARTMENT",
        "query": "Tìm căn hộ có thể cho thuê, gần siêu thị hoặc trung tâm thương mại, khoảng 4 tỷ",
        "purpose": "rental",
        "expected_facts": ["suitable_for_rental"],
        "expected_risks_should_avoid": ["low_data_confidence"],
    },
    {
        "case_id": "Q08_INVEST_GOOD_PRICE",
        "query": "Tìm bất động sản đầu tư giá tốt, không quá rủi ro, pháp lý ổn",
        "purpose": "investment",
        "expected_facts": ["suitable_for_investment", "good_value_by_area"],
        "expected_risks_should_avoid": ["possibly_overpriced", "need_legal_verification"],
    },
    {
        "case_id": "Q09_CAR_ACCESS",
        "query": "Tìm nhà có đường xe hơi, phù hợp gia đình, giá khoảng 7 tỷ",
        "purpose": "family",
        "expected_facts": ["good_car_access", "suitable_for_family"],
        "expected_risks_should_avoid": ["small_alley_access"],
    },
    {
        "case_id": "Q10_GENERAL_CLEAR_LEGAL",
        "query": "Tìm bất động sản pháp lý rõ, dữ liệu đầy đủ, giá hợp lý",
        "purpose": "general",
        "expected_facts": ["low_legal_risk", "high_data_confidence", "good_value_by_area"],
        "expected_risks_should_avoid": ["low_data_confidence", "need_legal_verification"],
    },
]

def run_test_query_suite_v4(test_cases, top_k=5, df=None):
    if df is None:
        df = bds_s456

    all_profiles = []
    all_results = []

    for case in test_cases:
        q = case["query"]
        profile, relax_used, recs = recommend_v4(q, df=df, top_k=top_k)

        profile_row = {
            "case_id": case["case_id"],
            "query": q,
            "expected_purpose": case.get("purpose"),
            "relax_level_used": relax_used,
            **{f"profile_{k}": v for k, v in profile.items() if k != "raw_query"},
        }
        all_profiles.append(profile_row)

        if len(recs) == 0:
            all_results.append({
                "case_id": case["case_id"],
                "query": q,
                "rank": None,
                "property_id": None,
                "final_score_v3": None,
                "note": "No result",
            })
            continue

        for rank, (_, row) in enumerate(recs.iterrows(), start=1):
            facts = _safe_list_456(row.get("symbolic_facts_v3", []))
            risks = _safe_list_456(row.get("risk_flags_v3", []))

            expected_facts = case.get("expected_facts", [])
            expected_risks_should_avoid = case.get("expected_risks_should_avoid", [])

            matched_expected_facts = sorted(list(set(facts) & set(expected_facts)))
            violated_risks = sorted(list(set(risks) & set(expected_risks_should_avoid)))

            all_results.append({
                "case_id": case["case_id"],
                "query": q,
                "rank": rank,
                "property_id": row.get("property_id"),
                "final_score_v3": row.get("final_score_v3"),
                "district": row.get("district"),
                "property_type": row.get("property_type"),
                "price_billion": row.get("price_billion"),
                "area_m2": row.get("area_m2"),
                "legal_class": row.get("legal_class"),
                "risk_score": row.get("risk_score"),
                "data_quality_score": row.get("data_quality_score"),
                "symbolic_facts_v3": facts,
                "risk_flags_v3": risks,
                "expected_facts": expected_facts,
                "matched_expected_facts": matched_expected_facts,
                "expected_risks_should_avoid": expected_risks_should_avoid,
                "violated_risks": violated_risks,
                "short_explanation_v4": row.get("short_explanation_v4"),
                "detailed_explanation_v4": row.get("detailed_explanation_v4"),
                "url": row.get("url"),
            })

    profiles_df = pd.DataFrame(all_profiles)
    results_df = pd.DataFrame(all_results)
    return profiles_df, results_df

profiles_v4, test_results_v4 = run_test_query_suite_v4(TEST_QUERY_SUITE_V4, top_k=5)

print("Số test cases:", len(TEST_QUERY_SUITE_V4))
print("Số dòng kết quả:", len(test_results_v4))

display(HTML("<h3>Parsed Profiles từ Test Queries</h3>"))
display(profiles_v4)

display(HTML("<h3>Kết quả Top-5 cho từng Test Query</h3>"))
display(test_results_v4[[
    "case_id", "rank", "property_id", "final_score_v3", "district", "property_type",
    "price_billion", "legal_class", "matched_expected_facts", "violated_risks", "short_explanation_v4"
]].head(30))

# Lưu Step 5
with open(STEP456_OUTPUT_DIR / "step5_test_query_suite_v4.json", "w", encoding="utf-8") as f:
    json.dump(TEST_QUERY_SUITE_V4, f, ensure_ascii=False, indent=2)

with pd.ExcelWriter(STEP456_OUTPUT_DIR / "step5_test_query_results_v4.xlsx", engine="openpyxl") as writer:
    profiles_v4.to_excel(writer, sheet_name="parsed_profiles", index=False)
    test_results_v4.to_excel(writer, sheet_name="topk_results", index=False)

test_results_v4.to_csv(STEP456_OUTPUT_DIR / "step5_test_query_results_v4.csv", index=False, encoding="utf-8-sig")
print("Đã lưu Step 5 tại:", STEP456_OUTPUT_DIR)



## Giải thích Step 5

Step 5 tạo bộ câu hỏi kiểm thử có chủ đích. Mỗi test case gồm câu truy vấn, mục đích kỳ vọng, facts kỳ vọng và risks nên tránh. Bộ này giúp kiểm tra hệ thống có trả về đúng loại BĐS hay không, ví dụ truy vấn kinh doanh thì nên kích hoạt `suitable_for_business`, truy vấn gia đình thì nên kích hoạt `suitable_for_family`.

Trong giai đoạn nghiên cứu thật, bộ test này nên được mở rộng và có người đánh nhãn thủ công. Ở mức prototype, nó đóng vai trò test set có kiểm soát để đánh giá nhanh chất lượng gợi ý.


In [ ]:

# ================================
# STEP 6 - SO SÁNH BASELINE VỚI MÔ HÌNH ĐỀ XUẤT
# ================================

import re
import subprocess
import sys

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

# Chuẩn hóa text fallback.
def _norm_eval_text(x):
    if "_normalize_query_text" in globals():
        return _normalize_query_text(x)
    import unicodedata
    text = str(x).lower().strip()
    text = text.replace("đ", "d")
    text = unicodedata.normalize("NFD", text)
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    return re.sub(r"\s+", " ", text)

def _tokenize_vi_simple(text):
    text = _norm_eval_text(text)
    tokens = re.findall(r"[a-z0-9]+", text)
    stop = set("va la cua co cho toi tim can nha dat bat dong san khoang tam duoi tren o tai quanh gan phu hop phap ly ro rang".split())
    return [t for t in tokens if len(t) >= 2 and t not in stop]

def _property_corpus_text(row):
    parts = []
    for c in ["title", "description", "raw_location", "district", "location_zone", "property_type", "legal_class", "price_band", "area_band"]:
        if c in row.index and not pd.isna(row.get(c)):
            parts.append(str(row.get(c)))
    for c in ["amenity_tags", "symbolic_facts_v3", "risk_flags_v3"]:
        vals = _safe_list_456(row.get(c, []))
        parts.extend([str(v) for v in vals])
    return _norm_eval_text(" ".join(parts))

# Chuẩn bị corpus dùng cho baseline keyword/TF-IDF.
bds_eval = bds_s456.copy().reset_index(drop=True)
bds_eval["_search_text"] = bds_eval.apply(_property_corpus_text, axis=1)

# -------------------------------
# Baseline 1: Keyword Search
# -------------------------------
def recommend_keyword_baseline(query, df=bds_eval, top_k=5):
    q_tokens = set(_tokenize_vi_simple(query))
    rows = []
    for _, row in df.iterrows():
        text_tokens = set(_tokenize_vi_simple(row.get("_search_text", "")))
        overlap = len(q_tokens & text_tokens)
        denom = max(len(q_tokens), 1)
        score = overlap / denom
        rows.append({
            "property_id": row.get("property_id"),
            "baseline_score": score,
            "method_score": score,
        })
    rec = pd.DataFrame(rows).sort_values("baseline_score", ascending=False).head(top_k)
    return rec.merge(df, on="property_id", how="left")

# -------------------------------
# Baseline 2: Hard Filter
# -------------------------------
def recommend_hard_filter_baseline(query, df=bds_eval, top_k=5):
    profile = parse_user_query_v3(query)
    temp = df.copy()

    mask = pd.Series([True] * len(temp), index=temp.index)

    if profile.get("budget_billion") is not None and "price_billion" in temp.columns:
        # Lọc cứng: không vượt quá 10% ngân sách.
        mask &= temp["price_billion"].isna() | (temp["price_billion"] <= profile["budget_billion"] * 1.10)

    if profile.get("district") is not None and "district" in temp.columns:
        # Lọc cứng: đúng quận/huyện, không lan truyền sang khu tương tự.
        mask &= temp["district"] == profile["district"]

    if profile.get("legal_required") and "legal_class" in temp.columns:
        mask &= temp["legal_class"] == "clear_ownership"

    if profile.get("bedrooms_min") is not None and "bedrooms" in temp.columns:
        mask &= temp["bedrooms"].isna() | (temp["bedrooms"] >= profile["bedrooms_min"])

    filtered = temp[mask].copy()

    # Nếu lọc cứng không có kết quả, fallback về toàn bộ dữ liệu và sắp theo giá hợp lý/chất lượng.
    if len(filtered) == 0:
        filtered = temp.copy()
        filtered["_hard_filter_fallback"] = 1
    else:
        filtered["_hard_filter_fallback"] = 0

    # Sort đơn giản theo price_reasonable_score và data_quality_score.
    for c in ["price_reasonable_score", "data_quality_score"]:
        if c not in filtered.columns:
            filtered[c] = 0.5
    filtered["baseline_score"] = (
        0.60 * filtered["price_reasonable_score"].fillna(0.5) +
        0.40 * filtered["data_quality_score"].fillna(0.5) -
        0.20 * filtered.get("_hard_filter_fallback", 0)
    )
    filtered["method_score"] = filtered["baseline_score"]
    return filtered.sort_values("baseline_score", ascending=False).head(top_k)

# -------------------------------
# Baseline 3: TF-IDF Search
# -------------------------------
_tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
_tfidf_matrix = _tfidf_vectorizer.fit_transform(bds_eval["_search_text"].fillna(""))

def recommend_tfidf_baseline(query, df=bds_eval, top_k=5):
    q = _norm_eval_text(query)
    q_vec = _tfidf_vectorizer.transform([q])
    sims = cosine_similarity(q_vec, _tfidf_matrix).ravel()
    idx = np.argsort(-sims)[:top_k]
    rec = df.iloc[idx].copy()
    rec["baseline_score"] = sims[idx]
    rec["method_score"] = rec["baseline_score"]
    return rec

# -------------------------------
# Proposed: Neuro-Symbolic/Fuzzy Rule + KG Facts
# -------------------------------
def recommend_proposed_ns(query, df=bds_eval, top_k=5):
    profile, relax_used, recs = recommend_v4(query, df=df, top_k=top_k)
    rec = recs.copy()
    if len(rec) > 0:
        rec["baseline_score"] = rec["final_score_v3"]
        rec["method_score"] = rec["final_score_v3"]
        rec["relax_level_used"] = relax_used
    return rec

# -------------------------------
# Pseudo relevance để đánh giá tự động
# Có thể thay bằng nhãn thủ công nếu có người đánh giá.
# -------------------------------
def _location_sim_eval(a, b):
    if b is None:
        return 0.60
    if "_district_similarity_local" in globals():
        try:
            return float(_district_similarity_local(a, b))
        except Exception:
            pass
    return 1.0 if a == b else 0.35

def heuristic_relevance_grade(row, profile, case=None):
    """
    Grade 0-3:
      0: không phù hợp
      1: hơi liên quan
      2: phù hợp
      3: rất phù hợp
    Lưu ý: đây là nhãn giả lập từ constraints, không phải nhãn người dùng thật.
    """
    score = 0.0
    max_score = 0.0

    # Ngân sách
    if profile.get("budget_billion") is not None:
        max_score += 1.0
        price = row.get("price_billion", np.nan)
        if pd.isna(price):
            score += 0.4
        else:
            ratio = price / max(profile["budget_billion"], 1e-9)
            if ratio <= 1.05:
                score += 1.0
            elif ratio <= 1.20:
                score += 0.75
            elif ratio <= 1.40:
                score += 0.45

    # Vị trí
    if profile.get("district") is not None:
        max_score += 1.0
        score += _location_sim_eval(row.get("district", "Không rõ"), profile["district"])

    # Pháp lý
    if profile.get("legal_required"):
        max_score += 1.0
        if row.get("legal_class") == "clear_ownership":
            score += 1.0
        elif row.get("legal_score", 0) >= 0.6:
            score += 0.5

    # Phòng ngủ
    if profile.get("bedrooms_min") is not None:
        max_score += 0.8
        bedrooms = row.get("bedrooms", np.nan)
        if pd.isna(bedrooms):
            score += 0.3
        elif bedrooms >= profile["bedrooms_min"]:
            score += 0.8
        elif bedrooms >= max(1, profile["bedrooms_min"] - 1):
            score += 0.4

    # Amenity
    desired_tags = profile.get("desired_amenities", [])
    if desired_tags:
        max_score += 0.8
        tags = set(_safe_list_456(row.get("amenity_tags", [])))
        matched = len(tags & set(desired_tags))
        score += 0.8 * matched / max(len(set(desired_tags)), 1)

    # Mục đích
    purpose = profile.get("purpose", "general")
    if purpose != "general":
        max_score += 1.2
        col = PURPOSE_SCORE_COL_V3.get(purpose) if "PURPOSE_SCORE_COL_V3" in globals() else None
        if col and col in row.index:
            v = row.get(col, 0)
            score += 1.2 * (0 if pd.isna(v) else float(v))
        else:
            facts = set(_safe_list_456(row.get("symbolic_facts_v3", [])))
            fact_name = f"suitable_for_{purpose}"
            score += 1.2 if fact_name in facts else 0.0

    # Expected facts / risks từ test case
    if case is not None:
        expected_facts = set(case.get("expected_facts", []))
        avoid_risks = set(case.get("expected_risks_should_avoid", []))
        facts = set(_safe_list_456(row.get("symbolic_facts_v3", [])))
        risks = set(_safe_list_456(row.get("risk_flags_v3", [])))

        if expected_facts:
            max_score += 1.0
            score += len(facts & expected_facts) / max(len(expected_facts), 1)

        if avoid_risks:
            max_score += 0.7
            violated = len(risks & avoid_risks)
            score += 0.7 if violated == 0 else max(0.0, 0.7 - 0.35 * violated)

    # Data quality/risk phụ trợ
    max_score += 0.7
    quality = row.get("data_quality_score", 0.5)
    risk = row.get("risk_score", 0.2)
    quality = 0.5 if pd.isna(quality) else float(quality)
    risk = 0.2 if pd.isna(risk) else float(risk)
    score += 0.4 * quality + 0.3 * (1 - min(max(risk, 0), 1))

    rel = score / max(max_score, 1e-9)
    if rel >= 0.78:
        return 3
    if rel >= 0.58:
        return 2
    if rel >= 0.35:
        return 1
    return 0

def dcg_at_k(grades, k):
    grades = np.asarray(grades[:k], dtype=float)
    if len(grades) == 0:
        return 0.0
    discounts = np.log2(np.arange(2, len(grades) + 2))
    return float(np.sum((2 ** grades - 1) / discounts))

def ndcg_at_k(grades, k):
    dcg = dcg_at_k(grades, k)
    ideal = dcg_at_k(sorted(grades, reverse=True), k)
    return 0.0 if ideal == 0 else dcg / ideal

def evaluate_method(method_name, recommender_fn, test_cases, top_k=5):
    metric_rows = []
    detail_rows = []

    for case in test_cases:
        query = case["query"]
        profile = parse_user_query_v3(query)
        recs = recommender_fn(query, top_k=top_k)

        if recs is None or len(recs) == 0:
            grades = []
        else:
            grades = []
            for rank, (_, row) in enumerate(recs.head(top_k).iterrows(), start=1):
                grade = heuristic_relevance_grade(row, profile, case=case)
                grades.append(grade)
                detail_rows.append({
                    "method": method_name,
                    "case_id": case["case_id"],
                    "query": query,
                    "rank": rank,
                    "property_id": row.get("property_id"),
                    "method_score": row.get("method_score", row.get("baseline_score", row.get("final_score_v3", np.nan))),
                    "relevance_grade": grade,
                    "is_relevant_grade2": int(grade >= 2),
                    "district": row.get("district"),
                    "property_type": row.get("property_type"),
                    "price_billion": row.get("price_billion"),
                    "legal_class": row.get("legal_class"),
                    "symbolic_facts_v3": row.get("symbolic_facts_v3", []),
                    "risk_flags_v3": row.get("risk_flags_v3", []),
                })

        relevant = [g >= 2 for g in grades]
        precision = sum(relevant) / top_k if top_k > 0 else 0.0
        hit_rate = 1.0 if any(relevant) else 0.0
        mrr = 0.0
        for i, rel in enumerate(relevant, start=1):
            if rel:
                mrr = 1.0 / i
                break
        ndcg = ndcg_at_k(grades, top_k)

        metric_rows.append({
            "method": method_name,
            "case_id": case["case_id"],
            "query": query,
            "Precision@K": precision,
            "HitRate@K": hit_rate,
            "MRR@K": mrr,
            "NDCG@K": ndcg,
            "avg_relevance_grade": np.mean(grades) if grades else 0.0,
            "num_results": len(grades),
        })

    return pd.DataFrame(metric_rows), pd.DataFrame(detail_rows)

METHODS = {
    "Keyword Search": recommend_keyword_baseline,
    "Hard Filter": recommend_hard_filter_baseline,
    "TF-IDF Search": recommend_tfidf_baseline,
    "Proposed Neuro-Symbolic": recommend_proposed_ns,
}

all_metrics = []
all_details = []
for method_name, fn in METHODS.items():
    m_df, d_df = evaluate_method(method_name, fn, TEST_QUERY_SUITE_V4, top_k=5)
    all_metrics.append(m_df)
    all_details.append(d_df)

metrics_by_case = pd.concat(all_metrics, ignore_index=True)
details_by_method = pd.concat(all_details, ignore_index=True)

metrics_summary = (
    metrics_by_case
    .groupby("method")
    .agg({
        "Precision@K": "mean",
        "HitRate@K": "mean",
        "MRR@K": "mean",
        "NDCG@K": "mean",
        "avg_relevance_grade": "mean",
        "num_results": "mean",
    })
    .reset_index()
    .sort_values("NDCG@K", ascending=False)
)

print("BẢNG SO SÁNH TRUNG BÌNH THEO PHƯƠNG PHÁP")
display(metrics_summary)

print("BẢNG METRIC THEO TỪNG TEST CASE")
display(metrics_by_case.head(40))

# Vẽ biểu đồ nếu plotly đã có.
try:
    import plotly.express as px
    metrics_long = metrics_summary.melt(
        id_vars="method",
        value_vars=["Precision@K", "HitRate@K", "MRR@K", "NDCG@K"],
        var_name="metric",
        value_name="value",
    )
    fig = px.bar(
        metrics_long,
        x="method",
        y="value",
        color="metric",
        barmode="group",
        title="So sánh các phương pháp khuyến nghị trên bộ test queries",
    )
    fig.update_layout(height=600, xaxis_title="Phương pháp", yaxis_title="Giá trị trung bình")
    fig.show()
except Exception as e:
    print("Không vẽ được biểu đồ:", e)

# Lưu Step 6
with pd.ExcelWriter(STEP456_OUTPUT_DIR / "step6_baseline_comparison_v4.xlsx", engine="openpyxl") as writer:
    metrics_summary.to_excel(writer, sheet_name="metrics_summary", index=False)
    metrics_by_case.to_excel(writer, sheet_name="metrics_by_case", index=False)
    details_by_method.to_excel(writer, sheet_name="details_by_method", index=False)

metrics_summary.to_csv(STEP456_OUTPUT_DIR / "step6_metrics_summary_v4.csv", index=False, encoding="utf-8-sig")
metrics_by_case.to_csv(STEP456_OUTPUT_DIR / "step6_metrics_by_case_v4.csv", index=False, encoding="utf-8-sig")
details_by_method.to_csv(STEP456_OUTPUT_DIR / "step6_details_by_method_v4.csv", index=False, encoding="utf-8-sig")

print("Đã lưu Step 6 tại:", STEP456_OUTPUT_DIR)



## Giải thích Step 6

Step 6 so sánh 4 phương pháp:

1. **Keyword Search**: đếm mức trùng từ khóa giữa câu hỏi và nội dung tin BĐS.
2. **Hard Filter**: lọc cứng theo ngân sách, quận, pháp lý và số phòng. Phương pháp này đơn giản nhưng dễ bị rỗng kết quả nếu dữ liệu không khớp tuyệt đối.
3. **TF-IDF Search**: biểu diễn câu hỏi và mô tả BĐS bằng vector TF-IDF, sau đó lấy cosine similarity.
4. **Proposed Neuro-Symbolic**: dùng parser nhu cầu, fuzzy scoring, symbolic facts, risk flags, data quality và rule evidence để xếp hạng.

Vì bộ dữ liệu hiện chưa có nhãn người dùng thật, notebook tạo `heuristic_relevance_grade` để đánh giá tự động theo constraints của từng test query. Khi làm báo cáo hoàn chỉnh, có thể thay nhãn giả lập này bằng đánh giá thủ công của chuyên gia/người dùng.


In [ ]:

# ================================
# NÉN OUTPUT STEP 4,5,6 ĐỂ TẢI VỀ
# ================================

import shutil

zip_path_456 = shutil.make_archive(str(STEP456_OUTPUT_DIR), "zip", root_dir=str(STEP456_OUTPUT_DIR))

print("File zip output Step 4,5,6:")
print(zip_path_456)

# Nếu chạy trên Colab, có thể mở dòng dưới để tải về:
# from google.colab import files
# files.download(zip_path_456)
